# Week 8 — From Images to Urban Clusters
### Collecting images of six Florida cities, turning them into feature vectors, and clustering them with Self-Organizing Maps

In weeks 3–6 you practised each step separately: collecting images through an API, preprocessing them and extracting VGG features, and clustering with a SOM.
Today you put the steps together into **one reproducible pipeline** that runs from raw web data to cluster labels you can map and interpret.

```
 1  COLLECT        ArchDaily (web scraping)  ·  Google satellite (API)  ·  Google Street View (API)
        │          Gainesville · Tallahassee · Jacksonville · Orlando · Tampa · Miami
        ▼
 2  DESCRIBE       4 "lenses" per image:  VGG16  |  semantic segmentation  |  edges  |  colour
        │          → long raw vector (.npz)  → standardise + weight + PCA → short embedding (CSV)
        ▼
 3  CLUSTER        SOM size sweep + quality metrics → final SOM → node & cluster label per image
                   → labels written back into the CSVs, SOM saved for later re-use
```

| Section | What you do | Saved in your Drive | Time |
|---|---|---|---|
| **0** Setup | mount Drive, settings, API key | folders | 5 min |
| **1** Data collection | scrape ArchDaily · plan sample points · download satellite & street view · interactive map | `01_images/` | 35 min |
| **2** Features | four feature extractors, visual checks, fusion, choosing the embedding size | `02_features/` | 40 min |
| **3** SOM | size sweep, quality metrics, clustering, maps, labels, saved model | `03_som/` + labels in CSVs | 30 min |
| **4** Wrap-up | check that everything is in Drive | – | 5 min |

**Before you start**
* `Runtime → Change runtime type → T4 GPU` (the notebook also runs on CPU, feature extraction is just ~5× slower).
* You need the **Google API key** from Week 3 (Street View Static API + Maps Static API enabled).
* Put `output_images.json` (the ArchDaily URLs scraped earlier in the course) in your Drive folder `Colab Notebooks/ML_for_Architects/Week8/`. If it is missing, the notebook uses a copy stored inside it.
* Run the cells **in order**. Every slow step saves its result to Drive, so if Colab disconnects you can re-run from the top and finished work is skipped (no image is downloaded or paid for twice).
* The notebook gets heavy with all the pictures and maps: before sharing it, use `Edit → Clear all outputs`.

---
# 0 · Setup

In [ ]:
# 0.1  Packages that Colab does not have by default (numpy, pandas, torch, transformers, scikit-learn, opencv… are already installed)
#      minisom is pinned: the saved SOM models (pickle) can only be re-loaded with the same version
%pip install -q minisom==2.3.6 folium pyproj beautifulsoup4

In [ ]:
# 0.2  Imports, Google Drive, helper functions
import os, sys, re, io, json, math, time, glob, random, pickle, base64, hashlib, itertools, threading, warnings, zlib
from pathlib import Path
from datetime import datetime
from collections import Counter
from urllib.parse import urljoin
import concurrent.futures as cf

import numpy as np
import pandas as pd
import requests
import folium
import matplotlib as mpl
import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import display, HTML
from tqdm.auto import tqdm

%config InlineBackend.figure_formats = ["jpeg"]

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)
plt.rcParams.update({"figure.dpi": 100, "axes.spines.top": False, "axes.spines.right": False})
pd.set_option("display.max_columns", 30)

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE_DIR = Path("/content/drive/MyDrive/Colab Notebooks/ML_for_Architects/Week8")
else:  # running on your own computer: everything goes into a local folder
    BASE_DIR = Path.cwd() / "ML_for_Architects" / "Week8"


def slugify(text):
    return re.sub(r"[^a-z0-9]+", "_", str(text).lower()).strip("_")


def load_rgb(path):
    """Open an image file and return it as an RGB PIL image (the file is closed afterwards)."""
    with Image.open(path) as im:
        return im.convert("RGB")


HTTP_HEADERS = {"User-Agent": "Mozilla/5.0 (compatible; ML-for-Architects-course/1.0; educational use)"}
MANIFEST_COLS = ["image_id", "city", "project", "article_url", "image_url", "relpath", "origin",
                 "lat", "lon", "locality", "location_source"]          # columns of the ArchDaily manifest


def save_jpeg(img, out_path):
    """Write to a temporary file first and rename it: an interrupted save never leaves a broken .jpg."""
    out_path.parent.mkdir(parents=True, exist_ok=True)
    tmp = out_path.with_name(out_path.name + ".part")
    img.save(tmp, "JPEG", quality=92)
    os.replace(tmp, out_path)


def load_arch_manifest():
    if ARCH_MANIFEST.exists():
        return pd.read_csv(ARCH_MANIFEST).reindex(columns=MANIFEST_COLS)
    return pd.DataFrame(columns=MANIFEST_COLS)


def show_image_grid(paths, titles=None, ncols=6, size=2.4, suptitle=None):
    """Show images in a grid. `paths` may contain None to leave a cell empty."""
    paths = list(paths)
    if not paths:
        print("(no images to show)")
        return
    nrows = math.ceil(len(paths) / ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * size, nrows * size * 0.95), squeeze=False)
    for ax in axes.ravel():
        ax.axis("off")
    for i, p in enumerate(paths):
        if p is None:
            continue
        ax = axes.ravel()[i]
        try:
            im = load_rgb(p)
            im.thumbnail((400, 400))
            ax.imshow(im)
        except Exception:
            ax.text(0.5, 0.5, "cannot open", ha="center", va="center")
        if titles is not None:
            ax.set_title(str(titles[i])[:45], fontsize=8)
    if suptitle:
        fig.suptitle(suptitle, fontsize=12)
    plt.tight_layout()
    plt.show()


def base_map(location=(28.2, -82.4), zoom=7):
    """An interactive map (OpenStreetMap + a satellite basemap you can switch on)."""
    m = folium.Map(location=location, zoom_start=zoom, tiles="OpenStreetMap", control_scale=True, prefer_canvas=True)
    folium.TileLayer(tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",
                     attr="Esri World Imagery", name="Esri satellite basemap").add_to(m)
    return m


def add_legend(m, items, title):
    rows = "".join(f'<div><span style="display:inline-block;width:11px;height:11px;border-radius:6px;'
                   f'background:{c};margin-right:6px"></span>{name}</div>' for name, c in items.items())
    m.get_root().html.add_child(folium.Element(
        f'<div style="position:fixed;bottom:25px;left:25px;z-index:9999;background:white;padding:8px 10px;'
        f'border:1px solid #bbb;border-radius:6px;font-size:12px"><b>{title}</b>{rows}</div>'))


print("Running on Colab" if IN_COLAB else "Running locally", "| project folder:", BASE_DIR)

### 0.3 Configuration
This is the **main cell to edit**. Each city is sampled inside one or more **urban units**: squares around a centre point.
The larger metro areas get several units (downtown + characteristic neighbourhoods or neighbouring municipalities), so each city's sample mixes different kinds of urban fabric instead of only the CBD.
Some units are separate municipalities inside the metro area — *Miami Beach* and *Coral Gables* (Miami), *Winter Park* (Orlando), *Jacksonville Beach* (Jacksonville) — they keep their own `unit` name in every table, so you can analyse them separately.

* `IMAGES_PER_CITY` — how many locations per city. **Satellite and street view use the same locations** (a satellite image and a street view image for each point), so the two data sets can be compared later.
* `DOWNLOAD_LIMIT_PER_CITY` — set to e.g. `5` for a quick first try, check the result, then set it back to `None` and re-run. Images already on disk are never downloaded (or paid for) twice.
* Cost: 6 cities × 150 points × 2 image types = **1,800 billed requests**. Google gives **10,000 free requests per month for each API**, and the metadata requests we use for planning are **free**.

In [ ]:
# 0.3  CONFIGURATION
#            unit name              centre lat  centre lon  square side (m)
CITY_UNITS = {
    "Gainesville": [
        ("downtown",            29.6516, -82.3248, 2000),
        ("midtown_uf",          29.6490, -82.3440, 2000),
    ],
    "Tallahassee": [
        ("downtown_capitol",    30.4390, -84.2810, 1600),
        ("collegetown_fsu",     30.4410, -84.2965, 1600),
        ("midtown",             30.4570, -84.2780, 1600),
    ],
    "Jacksonville": [
        ("downtown",            30.3290, -81.6590, 1800),
        ("riverside_avondale",  30.3080, -81.6900, 1800),
        ("san_marco",           30.3010, -81.6540, 1600),
        ("jax_beach",           30.2890, -81.3990, 1600),
    ],
    "Orlando": [
        ("downtown",            28.5420, -81.3790, 1800),
        ("mills50_thornton",    28.5530, -81.3640, 1600),
        ("college_park",        28.5700, -81.3910, 1600),
        ("winter_park",         28.5975, -81.3500, 1600),
    ],
    "Tampa": [
        ("downtown",            27.9480, -82.4590, 1600),
        ("ybor_city",           27.9610, -82.4370, 1600),
        ("hyde_park_soho",      27.9390, -82.4800, 1600),
        ("seminole_heights",    27.9960, -82.4600, 1600),
    ],
    "Miami": [
        ("downtown_brickell",   25.7680, -80.1950, 1800),
        ("wynwood",             25.8010, -80.1990, 1600),
        ("little_havana",       25.7680, -80.2200, 1600),
        ("miami_beach",         25.7820, -80.1340, 1600),
        ("coral_gables",        25.7490, -80.2620, 1600),
    ],
}
CITY_CODES = {"Gainesville": "GNV", "Tallahassee": "TLH", "Jacksonville": "JAX",
              "Orlando": "ORL", "Tampa": "TPA", "Miami": "MIA"}
CITY_COLORS = {"Gainesville": "#1f77b4", "Tallahassee": "#ff7f0e", "Jacksonville": "#2ca02c",
               "Orlando": "#d62728", "Tampa": "#9467bd", "Miami": "#e377c2", "Saint-Malo": "#7f7f7f"}

IMAGES_PER_CITY = 150                # locations per city (each gets 1 satellite + 1 street view image). Keep <= 200.
DOWNLOAD_LIMIT_PER_CITY = None       # e.g. 5 for a quick first try; None = download the whole plan
MAX_BILLED_REQUESTS_PER_RUN = 2500   # safety brake: max paid Google requests in one run of a download cell
ALLOW_BEYOND_FREE_QUOTA = False      # False = refuse to download if this month's logged requests would pass 10,000 per API

USE_SHARED_DATA = False              # True = don't download, use the image folder shared by your instructor
SHARED_IMG_DIR = "/content/drive/MyDrive/Week8_shared_data/01_images"

SEED = 42

In [ ]:
# 0.4  Folder structure in Drive (created automatically)
IMG_DIR = Path(SHARED_IMG_DIR) if USE_SHARED_DATA else BASE_DIR / "01_images"
ARCH_DIR, SAT_DIR, SV_DIR = IMG_DIR / "archdaily", IMG_DIR / "satellite", IMG_DIR / "streetview"
SOURCES = ["archdaily", "satellite", "streetview"]
SOURCE_DIRS = {"archdaily": ARCH_DIR, "satellite": SAT_DIR, "streetview": SV_DIR}

ARCH_MANIFEST = ARCH_DIR / "archdaily_manifest.csv"        # one row per ArchDaily image
PLAN_FILE = IMG_DIR / "sampling_plan.csv"                 # the chosen locations (shared by satellite & street view)
CAND_FILE = IMG_DIR / "sampling_candidates.csv"           # every grid point we checked, and why it was kept or not
API_USAGE_FILE = BASE_DIR / "api_usage.json"              # our own count of paid Google requests per month

FEAT_DIR = BASE_DIR / "02_features"
RAW_DIR, EMB_DIR = FEAT_DIR / "raw", FEAT_DIR / "embeddings"
FEAT_MODEL_DIR, FEAT_FIG_DIR = FEAT_DIR / "models", FEAT_DIR / "figures"
SOM_DIR = BASE_DIR / "03_som"
SOM_MODEL_DIR, SOM_FIG_DIR = SOM_DIR / "models", SOM_DIR / "figures"

if USE_SHARED_DATA and not IMG_DIR.exists():
    raise FileNotFoundError(f"Shared image folder not found: {IMG_DIR}\n"
                            "Open the shared link in Google Drive, choose 'Organize > Add shortcut' into My Drive, "
                            "and check SHARED_IMG_DIR.")
folders = [BASE_DIR, FEAT_DIR, RAW_DIR, EMB_DIR, FEAT_MODEL_DIR, FEAT_FIG_DIR, SOM_DIR, SOM_MODEL_DIR, SOM_FIG_DIR]
if not USE_SHARED_DATA:
    folders += [IMG_DIR, ARCH_DIR, SAT_DIR, SV_DIR]
for d in folders:
    d.mkdir(parents=True, exist_ok=True)
for src in SOURCES:
    for sub in (RAW_DIR, EMB_DIR):
        (sub / src).mkdir(parents=True, exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)
CITY_ORDER = list(CITY_UNITS) + ["Saint-Malo"]


def city_code(city):
    return CITY_CODES.get(city) or slugify(city)[:3].upper()


def city_color(city):
    return CITY_COLORS.get(city, "#555555")


# readable indicators (computed in Section 2) shown for each source in the plots of Sections 2 and 3
INT_COLS_BY_SOURCE = {
    "streetview": ["int_vegetation", "int_sky", "int_building", "int_road", "int_vehicle", "int_edge_density", "int_vertical_edges", "int_colourfulness"],
    "satellite": ["int_vegetation", "int_green_pixels", "int_grey_pixels", "int_blue_pixels", "int_edge_density", "int_brightness", "int_colourfulness"],
    "archdaily": ["int_interior", "int_wall", "int_sky", "int_vegetation", "int_building", "int_edge_density", "int_vertical_edges", "int_colourfulness"],
}
print(f"""Week8/
├── 01_images/      {'(shared, read-only) ' + str(IMG_DIR) if USE_SHARED_DATA else ''}
│   ├── archdaily/<city>/      scraped project images
│   ├── satellite/<city>/      Maps Static API
│   └── streetview/<city>/     Street View Static API
├── 02_features/    raw/ (long vectors, .npz) · embeddings/ (CSV per source & city) · models/
└── 03_som/         models/ · figures/""")

### 0.5 Your Google API key
**Recommended:** store the key as a Colab *secret* so it never appears in the notebook:
click the **🔑 key icon** in the left sidebar → *Add new secret* → name `GOOGLE_MAPS_API_KEY`, value = your key → switch on *Notebook access*.

Alternatively paste it into `GOOGLE_API_KEY` below — but then **never share this notebook**. (If you have no key, ArchDaily scraping still works and you can use `USE_SHARED_DATA = True`.)

In [ ]:
# 0.5  API key: pasted value  >  Colab secret "GOOGLE_MAPS_API_KEY"  >  environment variable
GOOGLE_API_KEY = ""


def find_api_key():
    if GOOGLE_API_KEY.strip():
        return GOOGLE_API_KEY.strip()
    if IN_COLAB:
        try:
            from google.colab import userdata
            return (userdata.get("GOOGLE_MAPS_API_KEY") or "").strip()
        except Exception as e:
            print("No Colab secret available:", type(e).__name__)
    return os.environ.get("GOOGLE_MAPS_API_KEY", "").strip()


API_KEY = find_api_key()
print("API key:", f"found (ends with …{API_KEY[-4:]})" if API_KEY else "NOT found -> Google downloads will be skipped")

---
# 1 · Data collection — three ways to get images

| | Method | Source | What you get | Cost |
|---|---|---|---|---|
| **1.1** | web scraping | ArchDaily project pages | photos & drawings of designed buildings | free — but be polite (robots.txt, delays) |
| **1.2** | API, *free* | Street View **metadata** | where panoramas exist | free, unlimited |
| **1.3** | API | Maps Static API (`maptype=satellite`) | top-down view of a location | 10,000 free / month, then $2 per 1,000 |
| **1.4** | API | Street View Static API | eye-level view of the same location | 10,000 free / month, then $7 per 1,000 |

At the end of this section your Drive contains three big folders — `archdaily/`, `satellite/`, `streetview/` — each with one sub-folder per city.

## 1.1 Web scraping: ArchDaily

Earlier in the course a scraper visited ArchDaily articles and saved every `<img>` URL into `output_images.json`. Scraped data is rarely clean, so before downloading anything we **inspect** it:

1. the file is not valid JSON — the scraper *appended* a new list on every run, so it contains `[...][...][...][...]`;
2. many URLs are not project pictures (logos, loading animations, 1-pixel placeholders, a Facebook tracking pixel);
3. ArchDaily stores every picture in several sizes (`thumb_jpg`, `medium_jpg`, `newsletter`, `large_jpg`…), so the same picture appears several times.

**Where to put the file:** run Section 0 first (it creates the folder), then drag `output_images.json` into *Google Drive › Colab Notebooks › ML_for_Architects › Week8* (the `ML_for_Architects` folder itself also works), or upload it with the 📁 *Files* panel on the left of Colab. If the file is not found, the notebook uses a copy of its URLs stored in the next cell.

In [ ]:
#@title 1.1a Backup copy of the URLs in output_images.json (used only if the file is not found)
FALLBACK_JSON_URLS = [
    'https://assets.adsttc.com/doodles/flat/logo-blue-full.svg',
    'https://assets.adsttc.com/doodles/flat/loader-blue.gif',
    'https://assets.adsttc.com/doodles/flat/logo-animation.gif',
    'https://images.adsttc.com/media/images/59f6/2f80/b22e/3829/dc00/0202/newsletter/BeFunky_Collage.jpg?1509306232',
    'data:image/gif;base64,R0lGODlhAQABAAAAACH5BAEKAAEALAAAAAABAAEAAAICTAEAOw==',
    'data:image/gif;base64,R0lGODlhAQABAIAAAAUEBAAAACwAAAAAAQABAAACAkQBADs=',
    'https://images.adsttc.com/media/images/59f6/2c13/b22e/3853/b800/014a/newsletter/ALTA_TOUR_ST_MALO_VUE_AXONO.jpg?1509305341',
    'https://images.adsttc.com/media/images/59f6/2d88/b22e/3853/b800/0151/newsletter/ALTA_TOUR_ST_MALO_VUE_MONTAGNE.jpg?1509305716',
    'https://images.adsttc.com/media/images/59f6/2cc7/b22e/3829/dc00/01f4/newsletter/01.17_Fac%CC%A7ade_Sud.jpg?1509305521',
    'https://images.adsttc.com/media/images/59f6/2d72/b22e/3829/dc00/01f8/newsletter/ALTA_TOUR_ST_MALO_VUE_MOKA.jpg?1509305691',
    'https://images.adsttc.com/media/images/59f6/2d41/b22e/3829/dc00/01f7/newsletter/ALTA_TOUR_ST_MALO_VUE_INTRA.jpg?1509305645',
    'https://images.adsttc.com/media/images/59f6/2d55/b22e/3853/b800/0150/medium_jpg/ALTA_TOUR_ST_MALO_VUE_MEDIATHEQUE.jpg?1509305667',
    'https://images.adsttc.com/media/images/59f6/2f80/b22e/3829/dc00/0202/medium_jpg/BeFunky_Collage.jpg?1509306232',
    'https://images.adsttc.com/media/images/59f6/2d14/b22e/3829/dc00/01f6/medium_jpg/ALTA_TOUR_ST_MALO_VUE_AXONO.jpg?1509305598',
    'https://images.adsttc.com/media/images/59f6/2bd9/b22e/3853/b800/0149/medium_jpg/01.12_Coupe_longitudinale.jpg?1509305278',
    'https://images.adsttc.com/media/images/59f6/2cf8/b22e/3829/dc00/01f5/medium_jpg/ALTA_TOUR_ST_MALO_VUE_ANITA_CONTI.jpg?1509305574',
    'https://images.adsttc.com/media/images/59f6/2d41/b22e/3829/dc00/01f7/medium_jpg/ALTA_TOUR_ST_MALO_VUE_INTRA.jpg?1509305645',
    'https://images.adsttc.com/media/images/59f6/2c13/b22e/3853/b800/014a/medium_jpg/ALTA_TOUR_ST_MALO_VUE_AXONO.jpg?1509305341',
    'https://images.adsttc.com/media/images/59f6/2d72/b22e/3829/dc00/01f8/medium_jpg/ALTA_TOUR_ST_MALO_VUE_MOKA.jpg?1509305691',
    'https://images.adsttc.com/media/images/59f6/2d29/b22e/3853/b800/014f/medium_jpg/ALTA_TOUR_ST_MALO_VUE_GARE.jpg?1509305624',
    'https://images.adsttc.com/media/images/59f6/2d88/b22e/3853/b800/0151/medium_jpg/ALTA_TOUR_ST_MALO_VUE_MONTAGNE.jpg?1509305716',
    'https://images.adsttc.com/media/images/59f6/2d9f/b22e/3829/dc00/01f9/medium_jpg/ALTA_TOUR_ST_MALO_VUE_REMPARTS.jpg?1509305739',
    'https://images.adsttc.com/media/images/59f6/2eb4/b22e/3829/dc00/0201/medium_jpg/SCHEMAS.jpg?1509306032',
    'https://images.adsttc.com/media/images/59f6/2ce0/b22e/3853/b800/014e/medium_jpg/01.18_Plan_masse_1_500.jpg?1509305548',
    'https://images.adsttc.com/media/images/59f6/2c3e/b22e/3853/b800/014b/medium_jpg/01.12_Coupe_longitudinale.jpg?1509305378',
    'https://images.adsttc.com/media/images/59f6/2c78/b22e/3853/b800/014c/medium_jpg/01.14_Fac%CC%A7ade_Nord.jpg?1509305442',
    'https://images.adsttc.com/media/images/59f6/2c5d/b22e/3829/dc00/01f2/medium_jpg/01.13_Coupe_transversale.jpg?1509305410',
    'https://images.adsttc.com/media/images/59f6/2bf9/b22e/3829/dc00/01f0/medium_jpg/01.13_Coupe_transversale.jpg?1509305310',
    'https://images.adsttc.com/media/images/59f6/2c92/b22e/3829/dc00/01f3/medium_jpg/01.15_Fac%CC%A7ade_Ouest.jpg?1509305468',
    'https://images.adsttc.com/media/images/59f6/2cc7/b22e/3829/dc00/01f4/medium_jpg/01.17_Fac%CC%A7ade_Sud.jpg?1509305521',
    'https://images.adsttc.com/media/images/59f6/2cad/b22e/3853/b800/014d/medium_jpg/01.16_Fac%CC%A7ade_Est.jpg?1509305495',
    'https://images.adsttc.com/media/images/59f6/2c1e/b22e/3829/dc00/01f1/medium_jpg/01.1_Sche%CC%81ma_situation.jpg?1509305367',
    'https://images.adsttc.com/media/images/59f6/2e79/b22e/3853/b800/0156/medium_jpg/plan_de_situation_2.jpg?1509305972',
    'https://images.adsttc.com/media/images/59f6/2e83/b22e/3829/dc00/0200/medium_jpg/plan_de_situation.jpg?1509305982',
    'https://images.adsttc.com/media/images/59f6/2db5/b22e/3829/dc00/01fa/medium_jpg/Binder1_Page_01.jpg?1509305765',
    'https://images.adsttc.com/media/images/59f6/2dca/b22e/3829/dc00/01fb/medium_jpg/Binder1_Page_02.jpg?1509305786',
    'https://images.adsttc.com/media/images/59f6/2dde/b22e/3853/b800/0152/medium_jpg/Binder1_Page_03.jpg?1509305807',
    'https://images.adsttc.com/media/images/59f6/2df3/b22e/3829/dc00/01fc/medium_jpg/Binder1_Page_04.jpg?1509305827',
    'https://images.adsttc.com/media/images/59f6/2e08/b22e/3853/b800/0153/medium_jpg/Binder1_Page_05.jpg?1509305848',
    'https://images.adsttc.com/media/images/59f6/2e1c/b22e/3829/dc00/01fd/medium_jpg/Binder1_Page_06.jpg?1509305869',
    'https://images.adsttc.com/media/images/59f6/2e31/b22e/3853/b800/0154/medium_jpg/Binder1_Page_07.jpg?1509305889',
    'https://images.adsttc.com/media/images/59f6/2e45/b22e/3829/dc00/01fe/medium_jpg/Binder1_Page_08.jpg?1509305910',
    'https://images.adsttc.com/media/images/59f6/2e5a/b22e/3853/b800/0155/medium_jpg/Binder1_Page_09.jpg?1509305931',
    'https://images.adsttc.com/media/images/59f6/2e6f/b22e/3829/dc00/01ff/medium_jpg/Binder1_Page_10.jpg?1509305951',
    'https://images.adsttc.com/media/image_maps/59f6/27b4/b22e/38cd/a900/0fcc/large/open-uri20171029-17960-lqzfgw.jpg?1509304386',
    'https://images.adsttc.com/media/images/60ba/8e9f/f91c/8166/5e00/005b/thumb_jpg/06_-_Les_Hauts_de_Sévigné_-_Cesson-Sévigné_-_a_LTA_HD_©_Charly_Broyez.jpg?1622838909',
    'https://images.adsttc.com/media/images/59f6/2d14/b22e/3829/dc00/01f6/thumb_jpg/ALTA_TOUR_ST_MALO_VUE_AXONO.jpg?1509305598',
    'https://www.facebook.com/tr?id=350870482443432&ev=PageView&noscript=1',
]
print(len(FALLBACK_JSON_URLS), "backup URLs available")

In [ ]:
# 1.1b Read output_images.json robustly
JSON_CANDIDATES = [BASE_DIR / "output_images.json", BASE_DIR.parent / "output_images.json",
                   BASE_DIR.parent.parent / "output_images.json", Path("output_images.json"),
                   Path("/content/output_images.json")]
json_path = next((p for p in JSON_CANDIDATES if p.exists()), None)


def read_concatenated_json(path):
    """Read a file that contains one or several JSON documents glued together, e.g. `[...][...]`.
    Damaged parts at the end are skipped: we keep everything that could be read."""
    text = Path(path).read_text(encoding="utf-8-sig")          # utf-8-sig also accepts a byte-order mark
    decoder, pos, items = json.JSONDecoder(), 0, []
    while pos < len(text):
        while pos < len(text) and (text[pos].isspace() or text[pos] == ","):
            pos += 1
        if pos >= len(text):
            break
        try:
            obj, pos = decoder.raw_decode(text, pos)
        except json.JSONDecodeError as e:
            print(f"  ! unreadable data at character {pos} ({e.msg}) -> keeping the {len(items)} records read so far")
            break
        items.extend(obj if isinstance(obj, list) else [obj])
    return items


raw_urls = []
if json_path is not None:
    try:
        json.loads(json_path.read_text(encoding="utf-8-sig"))
        print("The file is valid JSON.")
    except json.JSONDecodeError as e:
        print(f"json.load() fails ({e}) -> the file holds several JSON lists glued together.")
    records = read_concatenated_json(json_path)
    raw_urls = [r["image_url"] for r in records if isinstance(r, dict) and "image_url" in r]
    print(f"{len(raw_urls)} URLs read from {json_path}  ({len(set(raw_urls))} unique)")
if not raw_urls:
    print("⚠️  output_images.json not found (or empty) in", BASE_DIR, "-> using the backup copy stored in this notebook.")
    raw_urls = list(FALLBACK_JSON_URLS)

In [ ]:
# 1.1c What kind of URLs are these? Keep only real project pictures, one URL per picture.
ADSTTC_RE = re.compile(r"^https://images\.adsttc\.com/media/images/"
                       r"(?P<id>(?:[0-9a-f]{4}/){5}[0-9a-f]{4})/(?P<size>[a-z_]+)/(?P<name>[^?]+)")


def classify_url(url):
    if url.startswith("data:"):
        return "placeholder (data URI)"
    if "facebook.com" in url:
        return "tracking pixel"
    if "/doodles/" in url or url.lower().split("?")[0].endswith((".svg", ".gif")):
        return "logo / loading animation"
    if ADSTTC_RE.match(url):
        return "project picture"
    return "other"


def to_size(url, size="medium_jpg"):
    """The picture size is a folder in ArchDaily's URL: .../<picture id>/<size>/<file name>"""
    m = ADSTTC_RE.match(url)
    return url[:m.start("size")] + size + url[m.end("size"):]


def ad_image_id(url):
    return "AD_" + ADSTTC_RE.match(url).group("id").replace("/", "")


display(pd.Series(Counter(classify_url(u) for u in raw_urls), name="number of URLs").to_frame())

json_pictures = {}
for u in raw_urls:
    if ADSTTC_RE.match(u):
        json_pictures.setdefault(ad_image_id(u), to_size(u, "medium_jpg"))
print(f"{len(json_pictures)} distinct pictures after removing duplicates and size variants")

In [ ]:
# 1.1d Download the pictures (skips files that are already in Drive; save_jpeg and load_arch_manifest are in 0.2)
def download_image(url, out_path, min_side=100, timeout=30, retries=2):
    """Download one image and save it as an RGB JPEG. Returns a short status string."""
    out_path = Path(out_path)
    if out_path.exists() and out_path.stat().st_size > 0:
        return "cached"
    status = "error"
    for attempt in range(retries + 1):
        try:
            r = requests.get(url, headers=HTTP_HEADERS, timeout=timeout)
        except requests.RequestException as e:
            status = f"error: {type(e).__name__}"
            time.sleep(1 + attempt)
            continue
        if r.status_code != 200:
            status = f"error: HTTP {r.status_code}"
            if r.status_code in (429, 500, 502, 503, 504):
                time.sleep(2 + 2 * attempt)
                continue
            return status
        try:
            img = Image.open(io.BytesIO(r.content)).convert("RGB")
            if min(img.size) < min_side:
                return f"skipped: too small {img.size}"
            save_jpeg(img, out_path)
            return "downloaded"
        except (OSError, ValueError) as e:          # not an image, or the file could not be written
            return f"error: {type(e).__name__}"
    return status


def update_arch_manifest(new_rows):
    """Add rows to the ArchDaily manifest (one row per image, the newest row wins)."""
    rows = load_arch_manifest().to_dict("records") + list(new_rows)
    df = pd.DataFrame(rows, columns=MANIFEST_COLS).drop_duplicates("image_id", keep="last")
    on_disk = np.array([(IMG_DIR / str(p)).exists() for p in df.relpath], dtype=bool)
    df = df.loc[on_disk].sort_values(["city", "project", "image_id"])
    df.to_csv(ARCH_MANIFEST, index=False)
    return df.reset_index(drop=True)


JSON_CITY = "Saint-Malo"   # all pictures in output_images.json come from one article about a tower in Saint-Malo, France
JSON_PROJECT = "Semaphore mixed-use tower, Saint-Malo (FR)"

if USE_SHARED_DATA:
    print("USE_SHARED_DATA = True -> nothing to download.")
    json_rows = load_arch_manifest().query("origin == 'json'")
else:
    def _job(item):
        image_id, url = item
        rel = Path("archdaily") / JSON_CITY / f"{image_id}.jpg"
        return image_id, url, rel, download_image(url, IMG_DIR / rel)

    with cf.ThreadPoolExecutor(4) as ex:
        results = list(tqdm(ex.map(_job, json_pictures.items()), total=len(json_pictures), desc="ArchDaily (JSON)"))
    print(Counter(r[3] for r in results))
    new = [dict(image_id=i, city=JSON_CITY, project=JSON_PROJECT, article_url="output_images.json",
                image_url=u, relpath=rel.as_posix(), origin="json")
           for i, u, rel, st in results if st in ("downloaded", "cached")]
    if not new:
        print("⚠️  No picture could be downloaded (no internet, or images.adsttc.com refuses this machine). "
              "The notebook continues without them.")
    json_rows = update_arch_manifest(new).query("origin == 'json'")

show_image_grid([IMG_DIR / p for p in json_rows.relpath], titles=list(json_rows.image_id), ncols=8, size=2.0,
                suptitle=f"{len(json_rows)} pictures from output_images.json: renderings, elevations, sections, plans…")

### 1.1 (continued) Live scraping: Florida projects on ArchDaily

One article gives us only ~35 pictures, all from one French project. Let's scrape **projects located in our six Florida cities**, so ArchDaily becomes a third view of the same cities.

ArchDaily has a page per city (e.g. `https://www.archdaily.com/city/miami`, then `/page/2`, `/page/3`…). The scraper:
1. reads `robots.txt` and only visits pages the site allows (if `robots.txt` cannot be read, it stops);
2. collects the article links from the city pages (lectures, events and news are also listed there);
3. opens each article: a **project** page has a fact sheet with *Architects / Area / Year*, its own photos are in a list called `gallery-thumbs` (we ignore the "related projects" thumbnails elsewhere on the page), and a hidden data block (JSON-LD) often gives the project's **coordinates** — we use them to put the project on the map;
4. downloads up to `MAX_IMAGES_PER_PROJECT` pictures per project, evenly spaced through the gallery (photos first, drawings usually at the end), until the city has `TARGET_IMAGES_PER_CITY` pictures or no projects are left.

**What to expect:** ArchDaily publishes only a few projects per Florida city (Miami ~10, Tampa and Orlando ~4, Gainesville 2, Jacksonville and Tallahassee 1), so the counts are uneven: roughly 100 for Miami and Tampa, 10–30 for the small cities. The ArchDaily clusters in Section 3 will therefore reflect **projects and kinds of pictures** (exterior / interior / drawing) more than cities.

**Be polite:** one page per second, no redistribution — the images are © the architects and photographers, and we use them only for this exercise. A city that was scraped once is not visited again (set `RESCRAPE = True` to force it).
Websites sometimes block Colab's shared IP addresses. If that happens the cell says so and simply keeps the pictures it has, so the rest of the notebook still works.

In [ ]:
# 1.1e Live scraping of ArchDaily city pages (optional — set SCRAPE_LIVE = False to skip)
from bs4 import BeautifulSoup
from urllib.robotparser import RobotFileParser

SCRAPE_LIVE = True
RESCRAPE = False                 # True = visit ArchDaily again for cities that were already scraped
TARGET_IMAGES_PER_CITY = 100
MAX_IMAGES_PER_PROJECT = 25
MAX_CITY_PAGES = 3
REQUEST_DELAY_S = 1.0
AD_BASE = "https://www.archdaily.com"
ARTICLE_RE = re.compile(r"^https://www\.archdaily\.com/\d{3,8}/[a-z0-9-]+$")

# backup list of Florida project pages, used after the projects found on the city pages
BACKUP_PROJECT_URLS = {
    "Gainesville": [f"{AD_BASE}/1038014/university-of-florida-malachowsky-hall-for-data-science-and-information-technology-bohlin-cywinksi-jackson",
                    f"{AD_BASE}/547264/university-of-florida-clinical-translational-research-building-perkins-will"],
    "Tallahassee": [f"{AD_BASE}/278973/florida-state-university-william-h-johnston-building-gould-evans-architects"],
    "Jacksonville": [f"{AD_BASE}/86126/ad-classics-milam-residence-paul-rudolph"],
    "Orlando": [f"{AD_BASE}/1042378/resort-guest-amenities-building-poik-stanley-wilcox-architects",
                f"{AD_BASE}/439396/nemours-children-s-hospital-stanley-beaman-and-sears",
                f"{AD_BASE}/64270/ad-classics-walt-disney-world-swan-and-dolphin-resort-michael-graves"],
    "Tampa": [f"{AD_BASE}/957510/512-house-traction-architecture",
              f"{AD_BASE}/86741/tampa-covenant-church-alfonso-architects",
              f"{AD_BASE}/905862/form-of-wander-marc-fornes-theverymany",
              f"{AD_BASE}/52247/tampa-museum-of-art-stanley-saitowitz-natoma-architects"],
    "Miami": [f"{AD_BASE}/1036926/kiaora-strang",
              f"{AD_BASE}/986612/stewart-avenue-residence-brillhart-ar",
              f"{AD_BASE}/1018908/jade-alley-daniel-toole-architecture-plus-sb-architects",
              f"{AD_BASE}/603088/brillhart-house-brillhart-architecture",
              f"{AD_BASE}/902124/dilido-house-saota"],
}


class PoliteSession:
    """requests.Session that waits between requests and notices when the site blocks us."""

    def __init__(self, delay):
        self.session = requests.Session()
        self.session.headers.update(HTTP_HEADERS)
        self.delay, self.last, self.blocked = delay, 0.0, 0

    def get_html(self, url):
        wait = self.delay - (time.time() - self.last)
        if wait > 0:
            time.sleep(wait)
        self.last = time.time()
        try:
            r = self.session.get(url, timeout=30)
        except requests.RequestException as e:
            print(f"   ! {type(e).__name__}: {url}")
            return None
        head = r.text[:3000]
        if r.status_code in (403, 429, 503) or "Just a moment" in head or "cf-chl" in head:
            self.blocked += 1
            print(f"   ! blocked (HTTP {r.status_code}): {url}")
            return None
        if r.status_code != 200:
            print(f"   ! HTTP {r.status_code}: {url}")
            return None
        return r.text


def read_robots(sess):
    """Return the robots.txt rules, or None if robots.txt cannot be read (then we do not scrape)."""
    txt = sess.get_html(AD_BASE + "/robots.txt")
    if txt is None:
        return None
    rules = RobotFileParser()
    rules.parse(txt.splitlines())
    return rules


def article_links(html):
    """All links on a page that look like ArchDaily articles: https://www.archdaily.com/<number>/<slug>"""
    links = []
    for a in BeautifulSoup(html, "html.parser").find_all("a", href=True):
        url = urljoin(AD_BASE, a["href"]).split("?")[0].split("#")[0].rstrip("/")
        if ARTICLE_RE.match(url) and url not in links:
            links.append(url)
    return links


def city_article_urls(sess, robots, city, agent):
    """Article links from the city pages (page 1, 2, … read only when needed), then the backup list."""
    slug = slugify(city).replace("_", "-")
    seen = []
    for page in range(1, MAX_CITY_PAGES + 1):
        url = f"{AD_BASE}/city/{slug}" + (f"/page/{page}" if page > 1 else "")
        if not robots.can_fetch(agent, url):
            break
        html = sess.get_html(url)
        new = [u for u in article_links(html) if u not in seen] if html else []
        if not new:
            break
        seen += new
        yield from new
    yield from [u for u in BACKUP_PROJECT_URLS.get(city, []) if u not in seen]


def project_location(soup):
    """(locality, lat, lon) from the page's JSON-LD block, when ArchDaily provides it."""
    for s in soup.find_all("script", type="application/ld+json"):
        try:
            data = json.loads(s.string or "")
        except json.JSONDecodeError:
            continue
        for item in data if isinstance(data, list) else [data]:
            place = item.get("contentLocation") if isinstance(item, dict) else None
            if isinstance(place, dict):
                locality = (place.get("address") or {}).get("addressLocality")
                geo = place.get("geo") or {}
                try:
                    return locality, float(geo["latitude"]), float(geo["longitude"])
                except (KeyError, TypeError, ValueError):
                    return locality, None, None
    return None, None, None


def parse_project_page(html):
    """Return (is_project, title, gallery picture URLs, (locality, lat, lon)) for one ArchDaily article."""
    soup = BeautifulSoup(html, "html.parser")
    spec_keys = [s.get_text(" ", strip=True).rstrip(":").strip().lower() for s in soup.select(".afd-specs__key")]
    is_project = any(k.startswith("architect") for k in spec_keys)
    h1 = soup.find("h1")
    title = h1.get_text(" ", strip=True) if h1 else (soup.title.get_text(strip=True) if soup.title else "untitled")
    gallery = []
    for img in soup.select("ul.gallery-thumbs img"):
        for attr in ("src", "data-src"):
            u = img.get(attr) or ""
            if ADSTTC_RE.match(u) and to_size(u) not in gallery:
                gallery.append(to_size(u))
    return is_project, title, gallery, project_location(soup)


def pick_evenly(items, k):
    if len(items) <= k:
        return list(items)
    return [items[i] for i in dict.fromkeys(np.linspace(0, len(items) - 1, k).round().astype(int))]


def scrape_city(sess, robots, city, agent):
    rows, n_images = [], 0
    centre = CITY_UNITS[city][0][1:3]          # fallback location = centre of the city's first unit
    for url in city_article_urls(sess, robots, city, agent):
        if n_images >= TARGET_IMAGES_PER_CITY or sess.blocked >= 3:
            break
        if not robots.can_fetch(agent, url):
            print("   robots.txt disallows", url)
            continue
        page = sess.get_html(url)
        if page is None:
            continue
        is_project, title, gallery, (locality, lat, lon) = parse_project_page(page)
        if not is_project or not gallery:
            print(f"   - not a project page, skipped: {title[:75]}")
            continue
        k = min(MAX_IMAGES_PER_PROJECT, TARGET_IMAGES_PER_CITY - n_images)
        jobs = [(ad_image_id(u), u, Path("archdaily") / city / f"{ad_image_id(u)}.jpg") for u in pick_evenly(gallery, k)]
        with cf.ThreadPoolExecutor(4) as ex:
            statuses = list(ex.map(lambda j: download_image(j[1], IMG_DIR / j[2]), jobs))
        loc_source = "project page" if lat is not None else "city centre (approx.)"
        if lat is None:
            lat, lon = centre
        ok = [(i, u, rel) for (i, u, rel), st in zip(jobs, statuses) if st in ("downloaded", "cached")]
        rows += [dict(image_id=i, city=city, project=title, article_url=url, image_url=u, relpath=rel.as_posix(),
                      origin="live", lat=lat, lon=lon, locality=locality, location_source=loc_source)
                 for i, u, rel in ok]
        n_images += len(ok)
        where = f" — located in {locality}" if locality and city.lower() not in str(locality).lower() else ""
        print(f"   + {title[:70]}  ({len(ok)} of {len(gallery)} gallery pictures){where}")
    print(f"   = {n_images} pictures for {city} (target {TARGET_IMAGES_PER_CITY})")
    return rows


if USE_SHARED_DATA or not SCRAPE_LIVE:
    print("Live scraping skipped.")
else:
    already = set(load_arch_manifest().query("origin == 'live'").city)
    todo = [c for c in CITY_UNITS if RESCRAPE or c not in already]
    if not todo:
        print("All cities were scraped in an earlier run (set RESCRAPE = True to scrape again).")
    else:
        sess = PoliteSession(REQUEST_DELAY_S)
        robots = read_robots(sess)
        if robots is None:
            print("robots.txt could not be read -> we do not scrape (ArchDaily may be blocking this machine). "
                  "The notebook continues with the pictures it has.")
        else:
            for city in todo:
                if sess.blocked >= 3:
                    print("\nArchDaily is blocking this machine (common on Colab's shared IP addresses) -> "
                          "continuing with the pictures collected so far.")
                    break
                print(f"\n[{city}]")
                rows = scrape_city(sess, robots, city, HTTP_HEADERS["User-Agent"])
                update_arch_manifest(rows)          # saved after every city

In [ ]:
# 1.1f What did we collect?
arch_manifest = load_arch_manifest()
if len(arch_manifest):
    display(arch_manifest.groupby(["city", "project"]).size().rename("pictures").to_frame())
    paths, titles = [], []
    for city, g in arch_manifest.groupby("city", sort=False):
        g = g.sample(min(6, len(g)), random_state=SEED)
        paths += [IMG_DIR / p for p in g.relpath] + [None] * (6 - len(g))
        titles += [f"{city}\n{t[:28]}" for t in g.project] + [""] * (6 - len(g))
    show_image_grid(paths, titles, ncols=6, size=2.4, suptitle="ArchDaily: random pictures per city")
else:
    print("No ArchDaily pictures yet.")

## 1.2 Where to take the pictures? A sampling plan that avoids "missing" street views

In Week 3 we laid a **regular grid** over an area and requested an image at every grid point. For satellite images this always works, but for **street view many grid points fail**: the point is in the middle of a block, a parking lot, a lake or a private campus, and the API returns a grey "no imagery" picture — which we would still pay for.

**The fix: ask Google first, for free.** The *Street View metadata* endpoint answers "is there a panorama within 50 m of this point?" and returns the panorama's exact position, id and date. Metadata requests are **free and unlimited**. So the plan is:

```
for each urban unit of a city:
   1. lay a grid of candidate points every 110 m (in real metres) over each unit
   2. ask the metadata endpoint for every point                    (free; temporary errors are retried)
   3. drop:  no panorama nearby · user-uploaded photo spheres (not Google) · panoramas older than 2015
             · two grid points snapped to the same panorama
   4. randomly pick the quota of the unit, keeping >= 80 m between picked points
      (candidates whose grid point lies INSIDE a block, >= 15 m from its panorama, are picked first)
   5. camera heading: look from the panorama (on the street) towards the grid point (inside the block);
      if the grid point was on the street itself, probe 20 m around the panorama in 4 directions (free)
      to find the direction of the street, and look perpendicular to it (left or right side at random)
      -> the camera looks at the buildings along the street, not down the road
```
The plan is saved to Drive (`sampling_plan.csv`). **Satellite and street view use the same points**, so for every location we get a top view and a street view.
Every point gets an id like `MIA_wynwood_3fa2c1` whose last part is computed from the panorama and the heading: if you change the plan, changed points get new ids (and new images), and images of points that are no longer in the plan are moved to `01_images/_archive/`.
The candidate grid (fixed spacing) and the random order (fixed per unit) depend neither on `IMAGES_PER_CITY` nor on the other units: with 100 images per city you get the first 100 points of the 150-point plan, and moving or adding a unit leaves the points of the other units in place — so a change only downloads (and pays for) the points that are really new. Images that leave the plan are kept in `01_images/_archive/` and come back for free if they are needed again.

> Why metres and not degrees? One degree of longitude is ~111 km at the equator but ~97 km in Gainesville. We build each grid in a local projection centred on the unit, where distances are true metres (Web Mercator, used in Week 3, stretches distances by ~13 % at Florida's latitude).

In [ ]:
# 1.2a Tools: metric grid, distances, and the free Street View metadata request
from pyproj import Transformer

GOOGLE_STATIC_MAP_URL = "https://maps.googleapis.com/maps/api/staticmap"
GOOGLE_SV_URL = "https://maps.googleapis.com/maps/api/streetview"
GOOGLE_SV_META_URL = "https://maps.googleapis.com/maps/api/streetview/metadata"

GRID_SPACING_M = 110    # distance between candidate grid points (about 3 candidates per image at 150 images/city)
SNAP_RADIUS_M = 50      # search radius for a panorama around a grid point
MIN_SPACING_M = 80      # minimum distance between two selected points
MIN_PANO_YEAR = 2015    # ignore older panoramas
ON_STREET_M = 15        # grid point closer than this to its panorama = it was on the street itself
REBUILD_PLAN = False    # True = forget the saved plan and build a new one
RETRY_STATUSES = {"UNKNOWN_ERROR", "OVER_QUERY_LIMIT", "REQUEST_FAILED"}


def unit_grid(lat0, lon0, size_m, spacing_m):
    """Grid of (lat, lon) points with `spacing_m` metres between them, inside a square of side `size_m`."""
    local = f"+proj=aeqd +lat_0={lat0} +lon_0={lon0} +datum=WGS84 +units=m"   # true metres around the centre
    to_latlon = Transformer.from_crs(local, "EPSG:4326", always_xy=True)
    ticks = np.arange(-size_m / 2 + spacing_m / 2, size_m / 2, spacing_m)
    xx, yy = np.meshgrid(ticks, ticks)
    lon, lat = to_latlon.transform(xx.ravel(), yy.ravel())
    return np.column_stack([lat, lon])


def haversine_m(lat1, lon1, lat2, lon2):
    """Great-circle distance in metres (works with numbers or numpy arrays)."""
    lat1, lon1, lat2, lon2 = map(np.radians, (lat1, lon1, lat2, lon2))
    a = np.sin((lat2 - lat1) / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2) ** 2
    return 6371008.8 * 2 * np.arcsin(np.sqrt(a))


def bearing_deg(lat1, lon1, lat2, lon2):
    """Compass direction from point 1 to point 2 (0 = north, 90 = east)."""
    lat1, lon1, lat2, lon2 = map(math.radians, (lat1, lon1, lat2, lon2))
    x = math.sin(lon2 - lon1) * math.cos(lat2)
    y = math.cos(lat1) * math.sin(lat2) - math.sin(lat1) * math.cos(lat2) * math.cos(lon2 - lon1)
    return (math.degrees(math.atan2(x, y)) + 360) % 360


def sv_metadata(lat, lon, radius=None):
    """FREE request: is there an outdoor panorama within `radius` m of (lat, lon)? Temporary errors are retried."""
    params = {"location": f"{lat:.6f},{lon:.6f}", "radius": radius or SNAP_RADIUS_M, "source": "outdoor", "key": API_KEY}
    reply = {"status": "REQUEST_FAILED"}
    for attempt in range(4):
        try:
            reply = requests.get(GOOGLE_SV_META_URL, params=params, timeout=20).json()
        except Exception as e:
            reply = {"status": "REQUEST_FAILED", "error_message": type(e).__name__}
        if reply.get("status") not in RETRY_STATUSES:
            return reply
        time.sleep(1.5 * (attempt + 1))
    return reply


def offset_point(lat, lon, bearing, dist_m):
    """Point `dist_m` metres from (lat, lon) in compass direction `bearing`."""
    dlat = dist_m * math.cos(math.radians(bearing)) / 111_320
    dlon = dist_m * math.sin(math.radians(bearing)) / (111_320 * math.cos(math.radians(lat)))
    return lat + dlat, lon + dlon


def unit_square(lat0, lon0, size_m):
    """[[south, west], [north, east]] of a square of side size_m around a point (for maps)."""
    dlat = size_m / 2 / 111_320
    dlon = size_m / 2 / (111_320 * math.cos(math.radians(lat0)))
    return [[lat0 - dlat, lon0 - dlon], [lat0 + dlat, lon0 + dlon]]


if API_KEY and not USE_SHARED_DATA:      # quick test of the key on one point (free)
    test = sv_metadata(29.6516, -82.3248)
    print("metadata test (Gainesville downtown):", {k: test.get(k) for k in ("status", "date", "copyright", "pano_id")})
    if test.get("status") in ("REQUEST_DENIED", "INVALID_REQUEST"):
        print("⚠️  Google refused the key:", test.get("error_message", ""),
              "\n   -> check the key and that the Street View Static API is enabled (Week 3, Part 0).",
              "\n   Google downloads are switched off for this session; fix the key, then re-run 0.5 and this cell.")
        API_KEY = ""

In [ ]:
# 1.2b Build the plan for one city
def street_axes(lat, lon, pano_id, probe_m=20, radius_m=10):
    """Directions (0/45/90/135°) in which the street continues: probe 20 m away in each direction (FREE metadata).
    Along the street another Google panorama is found within 10 m; across the street (inside a block) there is none."""
    axes = []
    for axis in (0, 45, 90, 135):
        m = sv_metadata(*offset_point(lat, lon, axis, probe_m), radius=radius_m)
        if m.get("status") == "OK" and "Google" in str(m.get("copyright")) and m.get("pano_id") != pano_id:
            axes.append(axis)
    return axes


def plan_city(city, units, n_total):
    quotas = [n_total // len(units) + (1 if i < n_total % len(units) else 0) for i in range(len(units))]
    rows = []
    for (unit, lat0, lon0, size_m), quota in zip(units, quotas):
        pts = unit_grid(lat0, lon0, size_m, GRID_SPACING_M)
        with cf.ThreadPoolExecutor(16) as ex:
            metas = list(ex.map(lambda p: sv_metadata(p[0], p[1]), pts))
        for (glat, glon), m in zip(pts, metas):
            loc = m.get("location") or {}
            rows.append(dict(city=city, unit=unit, quota=quota, grid_lat=glat, grid_lon=glon,
                             status=m.get("status"), pano_id=m.get("pano_id"), pano_date=m.get("date"),
                             copyright=m.get("copyright"), lat=loc.get("lat"), lon=loc.get("lng"),
                             error=m.get("error_message")))
    cand = pd.DataFrame(rows)
    cand[["lat", "lon"]] = cand[["lat", "lon"]].astype(float)        # None (no panorama) -> NaN
    refused = cand[cand.status.isin(["REQUEST_DENIED", "INVALID_REQUEST", "OVER_DAILY_LIMIT"])]
    if len(refused):
        raise RuntimeError(f"Google refused the request: {refused.status.iloc[0]} – {refused.error.iloc[0]}\n"
                           "-> is the Street View Static API enabled for this key? (Week 3, Part 0)")

    # why is a candidate kept or dropped?
    year = pd.to_numeric(cand.pano_date.astype(str).str[:4], errors="coerce")
    google = cand.copyright.astype(str).str.contains("Google")
    cand["reason"] = "valid"
    cand.loc[cand.status != "OK", "reason"] = "no panorama nearby"
    cand.loc[cand.status.isin(RETRY_STATUSES), "reason"] = "request failed (retry later)"
    cand.loc[(cand.reason == "valid") & ~google, "reason"] = "user photo (not Google)"
    cand.loc[(cand.reason == "valid") & (year < MIN_PANO_YEAR), "reason"] = f"older than {MIN_PANO_YEAR}"
    valid = cand[cand.reason == "valid"]
    cand.loc[valid.index[valid.pano_id.duplicated()], "reason"] = "same panorama as another point"
    cand["inside_block"] = haversine_m(cand.lat.fillna(0).to_numpy(), cand.lon.fillna(0).to_numpy(),
                                       cand.grid_lat.to_numpy(), cand.grid_lon.to_numpy()) >= ON_STREET_M

    # random selection with a minimum spacing, unit by unit (pick_order = order of selection, used for quick tries)
    cand["selected"] = False
    cand["pick_order"] = -1
    picked_lat, picked_lon = [], []

    def try_pick(i):
        lat, lon = cand.at[i, "lat"], cand.at[i, "lon"]
        if picked_lat and haversine_m(np.array(picked_lat), np.array(picked_lon), lat, lon).min() < MIN_SPACING_M:
            return 0
        cand.at[i, "pick_order"] = len(picked_lat)
        picked_lat.append(lat); picked_lon.append(lon)
        cand.at[i, "selected"] = True
        return 1

    def random_order(g, key):
        """Random but fixed order (seeded by city/unit, not by the quota), candidates inside a block first."""
        rng = np.random.default_rng(SEED + zlib.crc32(f"{city}/{key}".encode()))
        return np.concatenate([rng.permutation(g.index[g.inside_block].to_numpy()),
                               rng.permutation(g.index[~g.inside_block].to_numpy())])

    for unit, g in cand[cand.reason == "valid"].groupby("unit", sort=False):
        need = int(g.quota.iloc[0])
        for i in random_order(g, unit):
            if need == 0:
                break
            need -= try_pick(i)
    missing = n_total - int(cand.selected.sum())     # a unit with too few panoramas -> borrow from the others
    for i in random_order(cand[(cand.reason == "valid") & ~cand.selected], "top-up"):
        if missing <= 0:
            break
        missing -= try_pick(i)

    # camera heading
    sel = cand[cand.selected].sort_values("pick_order").copy()
    on_street = sel[~sel.inside_block]
    with cf.ThreadPoolExecutor(16) as ex:
        axes = dict(zip(on_street.index, ex.map(lambda r: street_axes(r.lat, r.lon, r.pano_id), on_street.itertuples())))
    headings, rules = [], []
    for r in sel.itertuples():
        rng = np.random.default_rng(zlib.crc32(str(r.pano_id).encode()))   # same panorama -> same heading
        if r.inside_block:
            h, rule = bearing_deg(r.lat, r.lon, r.grid_lat, r.grid_lon), "towards the block"
        elif axes.get(r.Index):
            h, rule = (rng.choice(axes[r.Index]) + rng.choice([90.0, 270.0])) % 360, "perpendicular to the street"
        else:
            h, rule = float(rng.uniform(0, 360)), "random (street direction unknown)"
        headings.append(round(float(h), 1)); rules.append(rule)
    sel["heading"], sel["heading_rule"] = headings, rules
    tag = [hashlib.md5(f"{p}|{h:.1f}".encode()).hexdigest()[:6] for p, h in zip(sel.pano_id, sel.heading)]
    sel["point_id"] = [f"{city_code(city)}_{u}_{t}" for u, t in zip(sel.unit, tag)]
    sel["rank_in_unit"] = sel.groupby("unit").cumcount()
    cols = ["point_id", "city", "unit", "lat", "lon", "grid_lat", "grid_lon", "pano_id", "pano_date",
            "heading", "heading_rule", "rank_in_unit"]
    return sel[cols].reset_index(drop=True), cand


def plan_signature(city):
    """Fingerprint of the settings, so a saved plan is re-used only if nothing changed."""
    s = json.dumps([CITY_UNITS[city], IMAGES_PER_CITY, GRID_SPACING_M, SNAP_RADIUS_M, MIN_SPACING_M,
                    MIN_PANO_YEAR, ON_STREET_M, SEED])
    return hashlib.md5(s.encode()).hexdigest()[:10]

In [ ]:
# 1.2c Build (or re-use) the plan for all cities — uses only FREE metadata requests
def read_plan_file(path):
    return pd.read_csv(path, dtype={"signature": str}) if path.exists() else pd.DataFrame()


old_plan = pd.DataFrame() if REBUILD_PLAN else read_plan_file(PLAN_FILE)
old_cand = pd.DataFrame() if REBUILD_PLAN else read_plan_file(CAND_FILE)
PLAN, CANDIDATES = pd.DataFrame(), pd.DataFrame()

if USE_SHARED_DATA:
    PLAN, CANDIDATES = old_plan, old_cand
    print(f"Shared plan loaded: {len(PLAN)} points")
else:
    plans, cands = [], []

    def save_plan_files():
        """Write the plan of every city: the ones done in this run + the saved ones not reached yet."""
        done = {p.city.iloc[0] for p in plans if len(p)}
        rest_p = old_plan[~old_plan.city.isin(done)] if len(old_plan) else pd.DataFrame()
        rest_c = old_cand[~old_cand.city.isin(done)] if len(old_cand) else pd.DataFrame()
        all_p = [p for p in plans + [rest_p] if len(p)]
        all_c = [c for c in cands + [rest_c] if len(c)]
        if all_p:
            pd.concat(all_p, ignore_index=True).to_csv(PLAN_FILE, index=False)
        if all_c:
            pd.concat(all_c, ignore_index=True).to_csv(CAND_FILE, index=False)

    for city, units in CITY_UNITS.items():
        sig = plan_signature(city)
        if len(old_plan) and "signature" in old_plan and ((old_plan.city == city) & (old_plan.signature == sig)).any():
            plans.append(old_plan[old_plan.city == city])
            if len(old_cand):
                cands.append(old_cand[old_cand.city == city])
            print(f"{city:13s} saved plan re-used ({(old_plan.city == city).sum()} points)")
            continue
        if not API_KEY:
            print(f"{city:13s} no saved plan and no (working) API key -> skipped")
            continue
        t0 = time.time()
        try:
            sel, cand = plan_city(city, units, IMAGES_PER_CITY)
        except RuntimeError as e:
            print(f"{city:13s} ⚠️ {e}\n              -> Google downloads are switched off for this session; "
                  "cities with a saved plan are still re-used.")
            API_KEY = ""
            continue
        sel["signature"] = sig
        plans.append(sel)
        cands.append(cand)
        rules = sel.heading_rule.value_counts(normalize=True)
        print(f"{city:13s} {len(cand):4d} grid points checked -> {(cand.reason == 'valid').sum():4d} valid "
              f"-> {len(sel):4d} selected ({time.time() - t0:.0f} s, free) | heading: "
              + ", ".join(f"{k} {v:.0%}" for k, v in rules.items()))
        save_plan_files()                     # after every new city, so nothing is lost if a later city fails
    save_plan_files()
    if plans:
        PLAN = pd.concat(plans, ignore_index=True)
        CANDIDATES = pd.concat([c for c in cands if len(c)], ignore_index=True) if any(len(c) for c in cands) else pd.DataFrame()

if len(CANDIDATES):
    table = CANDIDATES.groupby(["city", "unit", "reason"]).size().unstack(fill_value=0)
    table["SELECTED"] = CANDIDATES.groupby(["city", "unit"]).selected.sum()
    display(table)

In [ ]:
# 1.2d Look at the plan on a map BEFORE paying for any image
REASON_COLORS = {"valid": "#4c9be8", "no panorama nearby": "#9e9e9e", "user photo (not Google)": "#f39c12",
                 f"older than {MIN_PANO_YEAR}": "#8d6e63", "same panorama as another point": "#cfcfcf",
                 "request failed (retry later)": "#000000"}


SHOW_CANDIDATES_FOR = "Miami"     # city whose rejected grid points are drawn ("ALL" = every city, heavier)


def plan_map(plan, cand):
    m = base_map()
    for city, units in CITY_UNITS.items():
        fg = folium.FeatureGroup(name=city)
        for unit, lat0, lon0, size_m in units:
            folium.Rectangle(unit_square(lat0, lon0, size_m), color=city_color(city), weight=2, dash_array="6",
                             fill=False, tooltip=f"{city} / {unit}").add_to(fg)
        if len(cand) and SHOW_CANDIDATES_FOR in ("ALL", city):
            for r in cand[(cand.city == city) & ~cand.selected.astype(bool)].itertuples():
                folium.CircleMarker([r.grid_lat, r.grid_lon], radius=2, weight=0, fill=True, fill_opacity=0.8,
                                    color=REASON_COLORS.get(r.reason, "#999"), tooltip=r.reason).add_to(fg)
        for r in plan[plan.city == city].itertuples():
            folium.CircleMarker([r.lat, r.lon], radius=4, weight=1, fill=True, fill_opacity=0.9, color=city_color(city),
                                tooltip=f"{r.point_id} · panorama {r.pano_date}").add_to(fg)
        fg.add_to(m)
    folium.LayerControl(collapsed=False).add_to(m)
    add_legend(m, {"selected point (coloured by city)": "#333333", **REASON_COLORS}, "grid candidates")
    return m


if len(PLAN):
    display(plan_map(PLAN, CANDIDATES))
else:
    print("No plan yet.")

**Read the map:** zoom into a city. Dashed squares are the urban units, coloured dots the selected points; for the city in `SHOW_CANDIDATES_FOR` (Miami by default) the small dots are the rejected grid points: grey = no panorama (water, inside large blocks, parks), orange = user-uploaded photos that we skip.
✏️ *Try:* change a unit's centre or size in `CITY_UNITS`, re-run 0.3 → 1.2c → 1.2d. Only the changed city is planned again (free), and only the changed unit gets new points: if you then download again, only those cost requests.

## 1.3 Satellite images (Maps Static API)

We request a **640 × 640 px** image at **zoom 19** centred on each planned point. At Florida's latitude one pixel is ≈ 0.26 m, so each image covers ≈ **170 m × 170 m** — about one city block.
Every download is counted: `MAX_BILLED_REQUESTS_PER_RUN` stops a run before it gets expensive, `api_usage.json` in your Drive keeps a monthly total of what this notebook requested, and with `ALLOW_BEYOND_FREE_QUOTA = False` a download is refused if it would push this month's total past the free 10,000 (your Google Cloud console shows the official number).

In [ ]:
# 1.3a Download helpers shared by satellite and street view
SAT_ZOOM, SAT_SIZE = 19, 640
SV_SIZE, SV_FOV, SV_PITCH = "640x480", 90, 10
FREE_PER_MONTH = 10_000


def logged_usage():
    log = json.loads(API_USAGE_FILE.read_text()) if API_USAGE_FILE.exists() else {}
    return log, datetime.now().strftime("%Y-%m")


class ApiBudget:
    """Counts paid requests in one run and refuses more than `max_per_run`."""

    def __init__(self, max_per_run):
        self.max_per_run, self.used, self.lock = max_per_run, Counter(), threading.Lock()

    def take(self, kind):
        with self.lock:
            if sum(self.used.values()) >= self.max_per_run:
                return False
            self.used[kind] += 1
            return True

    def save_and_report(self):
        log, month = logged_usage()
        for kind, n in self.used.items():
            log.setdefault(month, {})
            log[month][kind] = log[month].get(kind, 0) + n
        API_USAGE_FILE.write_text(json.dumps(log, indent=1))
        print(f"Paid requests in this run: {dict(self.used) or 0} | this month from this notebook: "
              f"{log.get(month, {})} (free quota: {FREE_PER_MONTH:,} per API per month)")


def sat_footprint_m(lat, zoom=SAT_ZOOM, size_px=SAT_SIZE):
    """Ground width (m) of a Static Maps image: Web Mercator metres-per-pixel x pixels."""
    return 156543.03392 * math.cos(math.radians(lat)) / 2 ** zoom * size_px


def fetch_google_image(url, params, out_path, kind, budget):
    """One paid request -> JPEG on disk. Files that already exist are skipped (and not paid for)."""
    if out_path.exists() and out_path.stat().st_size > 0:
        return "cached"
    if not budget.take(kind):
        return "skipped: run budget reached"
    try:
        r = requests.get(url, params={**params, "key": API_KEY}, timeout=30)
    except requests.RequestException as e:
        return f"error: {type(e).__name__}"
    if r.status_code != 200 or not r.headers.get("content-type", "").startswith("image"):
        return f"error: HTTP {r.status_code} {r.text[:150]}"
    try:
        save_jpeg(Image.open(io.BytesIO(r.content)).convert("RGB"), out_path)
    except (OSError, ValueError) as e:          # not an image, or the file could not be written
        return f"error: {type(e).__name__} while saving"
    return "downloaded"


def manifest_path(kind):
    return SOURCE_DIRS[kind] / f"{kind}_manifest.csv"


def write_google_manifest(kind):
    man = PLAN[["point_id", "city", "unit", "lat", "lon", "grid_lat", "grid_lon", "pano_id", "pano_date",
                "heading", "heading_rule"]].copy()
    man.insert(0, "image_id", man.point_id)
    man["relpath"] = [f"{kind}/{c}/{p}.jpg" for c, p in zip(man.city, man.point_id)]
    man["downloaded"] = [(IMG_DIR / p).exists() for p in man.relpath]
    man.to_csv(manifest_path(kind), index=False)
    return man


def sync_folder_with_plan(kind):
    """Images that are not in the current plan (e.g. after changing a unit) leave the city folders for
    01_images/_archive/; archived images that ARE in the plan again come back (no need to pay twice)."""
    keep = {f"{c}/{p}.jpg" for c, p in zip(PLAN.city, PLAN.point_id)}
    planned_cities = set(PLAN.city)              # folders of cities without a plan in this session are left alone
    archive = IMG_DIR / "_archive" / kind
    moved = restored = 0
    for f in SOURCE_DIRS[kind].glob("*/*.jpg"):
        if f.parent.name not in planned_cities:
            continue
        rel = f"{f.parent.name}/{f.name}"
        if rel not in keep:
            (archive / rel).parent.mkdir(parents=True, exist_ok=True)
            os.replace(f, archive / rel)
            moved += 1
    for rel in keep:
        if (archive / rel).exists() and not (SOURCE_DIRS[kind] / rel).exists():
            (SOURCE_DIRS[kind] / rel).parent.mkdir(parents=True, exist_ok=True)
            os.replace(archive / rel, SOURCE_DIRS[kind] / rel)
            restored += 1
    for f in SOURCE_DIRS[kind].glob("*/*.part"):
        f.unlink()
    if moved:
        print(f"{moved} {kind} images from an older plan were moved to {archive}")
    if restored:
        print(f"{restored} {kind} images were taken back from {archive} (not downloaded again)")


AUTH_ERROR_WORDS = ("HTTP 401", "HTTP 403", "API key", "not authorized", "REQUEST_DENIED", "Billing")


def run_google_downloads(kind):
    if USE_SHARED_DATA:
        print("USE_SHARED_DATA = True -> nothing to download.")
        return
    if not API_KEY or PLAN.empty:
        print("No (working) API key or no sampling plan -> skipped.")
        return
    folder = SOURCE_DIRS[kind]
    sync_folder_with_plan(kind)
    # a limited download takes the first points of EVERY unit (round robin), not one unit
    plan = PLAN.sort_values(["city", "rank_in_unit", "unit"], key=lambda s: s.map(list(CITY_UNITS).index)
                            if s.name == "city" else s)
    if DOWNLOAD_LIMIT_PER_CITY is not None:
        plan = plan.groupby("city", sort=False).head(DOWNLOAD_LIMIT_PER_CITY)
    jobs = list(plan.itertuples())
    todo = [r for r in jobs if not (folder / r.city / f"{r.point_id}.jpg").exists()]
    print(f"{kind}: {len(jobs)} images wanted, {len(jobs) - len(todo)} already in Drive, {len(todo)} to download")

    log, month = logged_usage()
    so_far = log.get(month, {}).get(kind, 0)
    if todo and not ALLOW_BEYOND_FREE_QUOTA and so_far + len(todo) > FREE_PER_MONTH:
        print(f"⚠️  This month this notebook already made {so_far} {kind} requests; {len(todo)} more would pass the "
              f"free {FREE_PER_MONTH:,}. Nothing downloaded (set ALLOW_BEYOND_FREE_QUOTA = True to continue anyway).")
        write_google_manifest(kind)
        return

    budget = ApiBudget(MAX_BILLED_REQUESTS_PER_RUN)

    def work(r):
        out = folder / r.city / f"{r.point_id}.jpg"
        if kind == "satellite":
            params = {"center": f"{r.lat:.6f},{r.lon:.6f}", "zoom": SAT_ZOOM, "size": f"{SAT_SIZE}x{SAT_SIZE}",
                      "maptype": "satellite"}
            return fetch_google_image(GOOGLE_STATIC_MAP_URL, params, out, kind, budget)
        params = {"pano": r.pano_id, "size": SV_SIZE, "heading": r.heading, "pitch": SV_PITCH, "fov": SV_FOV,
                  "return_error_code": "true"}
        return fetch_google_image(GOOGLE_SV_URL, params, out, kind, budget)

    try:
        statuses = []
        for r in todo[:3]:                       # a few test requests first: stop at once if the key is refused
            statuses.append(work(r))
            if any(w in statuses[-1] for w in AUTH_ERROR_WORDS):
                print("Google refused the request ->", statuses[-1],
                      "\nCheck the API key and that the API is enabled (Week 3, Part 0).")
                return
        if len(todo) > 3:
            with cf.ThreadPoolExecutor(8) as ex:
                statuses += list(tqdm(ex.map(work, todo[3:]), total=len(todo) - 3, desc=kind))
        if statuses:
            print(Counter(s.split(":")[0] for s in statuses))
            errors = [s for s in statuses if s.startswith("error")]
            if errors:
                print("example error:", errors[0], "-> re-run the cell to retry the failed images")
            if any(s.startswith("skipped") for s in statuses):
                print(f"Run budget of {MAX_BILLED_REQUESTS_PER_RUN} paid requests reached -> re-run the cell to continue.")
    finally:
        man = write_google_manifest(kind)
        print(f"{int(man.downloaded.sum())} of {len(man)} planned {kind} images are in Drive")
        budget.save_and_report()


def show_city_samples(kind, n=6):
    if not manifest_path(kind).exists():
        print("No manifest yet.")
        return
    man = pd.read_csv(manifest_path(kind))
    man = man[man.downloaded.astype(bool)]
    paths, titles = [], []
    for city, g in man.groupby("city", sort=False):
        g = g.sample(min(n, len(g)), random_state=SEED)
        paths += [IMG_DIR / p for p in g.relpath] + [None] * (n - len(g))
        titles += [f"{city} · {u}" for u in g.unit] + [""] * (n - len(g))
    show_image_grid(paths, titles, ncols=n, size=2.4, suptitle=f"{kind}: random images per city")

In [ ]:
# 1.3b Download satellite images
run_google_downloads("satellite")
show_city_samples("satellite")

## 1.4 Street view images (Street View Static API)

We ask for the **panorama id** found during planning (so the image is guaranteed to exist), a **640 × 480** view, field of view 90°, camera tilted up by 10° (to see facades rather than asphalt), and the heading chosen in the plan.
`return_error_code=true` makes Google return an error instead of a grey "no imagery" picture if something goes wrong.

In [ ]:
# 1.4 Download street view images
run_google_downloads("streetview")
show_city_samples("streetview")

## 1.5 Interactive map of the collection

Every **circle** is a street view camera (the short line shows where it looks); every **square** is the ground covered by the satellite image at that point; every **star** is an ArchDaily project (at the coordinates given on its page, or at the city centre when the page has none — projects at the same spot share one star, click it to see all of them).
Use the layer control (top right) to switch cities and sources. Click a project, or — for the city in `SHOW_THUMBNAILS_FOR` — a circle or square, to see the picture (thumbnails are embedded in the map, so showing all cities at once makes the notebook heavy).

In [ ]:
# 1.5 Interactive map: one point per image
SHOW_THUMBNAILS_FOR = "Miami"     # a city name, "ALL" (heavy) or None


def thumb_html(path, width=160):
    try:
        with Image.open(path) as im:
            im = im.convert("RGB")
            im.thumbnail((width, width))
            buf = io.BytesIO()
            im.save(buf, "JPEG", quality=60)
    except Exception:
        return "(image not readable)"
    return f'<img src="data:image/jpeg;base64,{base64.b64encode(buf.getvalue()).decode()}" width="{width}">'


def collection_map():
    m = base_map()
    mans = {k: pd.read_csv(manifest_path(k)) for k in ("satellite", "streetview") if manifest_path(k).exists()}
    for kind, man in mans.items():
        man = man[man.downloaded.astype(bool)]
        for city in CITY_UNITS:
            sub = man[man.city == city]
            if sub.empty:
                continue
            thumbs = SHOW_THUMBNAILS_FOR in ("ALL", city)
            fg = folium.FeatureGroup(name=f"{city} · {kind}", show=True)
            for r in sub.itertuples():
                popup = folium.Popup(thumb_html(IMG_DIR / r.relpath) + f"<br><b>{r.point_id}</b> · {r.pano_date}",
                                     max_width=200) if thumbs else None
                if kind == "satellite":
                    folium.Rectangle(unit_square(r.lat, r.lon, sat_footprint_m(r.lat)), color=city_color(city),
                                     weight=1, fill=True, fill_opacity=0.12, tooltip=r.point_id, popup=popup).add_to(fg)
                else:
                    folium.PolyLine([(r.lat, r.lon), offset_point(r.lat, r.lon, r.heading, 35)], color="black", weight=2).add_to(fg)
                    folium.CircleMarker([r.lat, r.lon], radius=4, color=city_color(city), fill=True, fill_opacity=0.9,
                                        weight=1, tooltip=f"{r.point_id} (heading {r.heading:.0f}°)", popup=popup).add_to(fg)
            fg.add_to(m)
    arch = load_arch_manifest().dropna(subset=["lat", "lon"])
    if len(arch):
        fg = folium.FeatureGroup(name="ArchDaily projects", show=True)
        for (lat, lon), here in arch.groupby([arch.lat.round(5), arch.lon.round(5)]):   # projects at the same spot share one star
            parts, names = [], []
            for project, g in here.groupby("project"):
                r = g.iloc[0]
                parts.append(thumb_html(IMG_DIR / r.relpath, 140) + f"<br><b>{project}</b><br>{len(g)} pictures · "
                             f"location: {r.location_source}")
                names.append(project)
            folium.Marker([lat, lon], tooltip=" | ".join(names), popup=folium.Popup("<hr>".join(parts), max_width=180),
                          icon=folium.Icon(color="darkpurple", icon="star")).add_to(fg)
        fg.add_to(m)
    folium.LayerControl(collapsed=True).add_to(m)
    add_legend(m, {c: city_color(c) for c in CITY_UNITS}, "cities")
    return m


if any(manifest_path(k).exists() for k in ("satellite", "streetview")) or ARCH_MANIFEST.exists():
    display(collection_map())
else:
    print("Nothing downloaded yet.")

In [ ]:
# 1.6 Summary: images per source and city (= sub-folders in Drive)
def collection_summary():
    rows = []
    for src in SOURCES:
        for city_dir in sorted(p for p in (IMG_DIR / src).glob("*") if p.is_dir()):
            rows.append(dict(source=src, city=city_dir.name, images=len(list(city_dir.glob("*.jpg")))))
    if not rows:
        print("No images yet.")
        return
    df = pd.DataFrame(rows).pivot_table(index="city", columns="source", values="images", fill_value=0, aggfunc="sum")
    df.loc["TOTAL"] = df.sum()
    display(df)


collection_summary()

---
# 2 · Turning images into numbers

A clustering algorithm cannot look at pictures; it compares **vectors**. How we turn an image into a vector decides what "similar" will mean.
We describe every image through four complementary **lenses**:

| Lens | Method | Captures | Length |
|---|---|---|---|
| **VGG16** | CNN pre-trained on ImageNet (as in Week 4), average of each of the 512 channels of the last convolution block | textures, objects, "style" — rich but hard to interpret | 512 |
| **Segmentation** | **SegFormer** (an encoder–decoder network, like U-Net), trained on the ADE20K scene dataset: every pixel gets one of 150 classes | *what* is in the image: % sky, building, tree, road, car, water… | 150 |
| **Edges** | Canny edges (Week 4) → edge density on an 8 × 8 grid + histogram of edge orientations (weighted by edge strength) | structure: busy vs calm, vertical facades vs horizontal lines | 81 |
| **Colour** | HSV histograms, colour statistics, colourfulness, 4 × 4 grid of average colours | palette and its layout (blue sky on top, green below…) | 89 |

Concatenated, each image becomes a **832-number raw vector**. We store these raw vectors in compressed `.npz` files (binary; the long vector, the four blocks and the name of every column), then **standardise, weight and compress them with PCA** into a short embedding that we save as CSV — one CSV per source and per city.

> *Why not a U-Net?* U-Net is the classic encoder–decoder for segmentation, but there is no ready-made U-Net trained on street scenes. SegFormer has the same encoder–decoder idea, is small (3.8 M parameters) and is pre-trained on ADE20K, whose classes (building, sky, tree, palm, road, sidewalk, car…) are exactly what urban analysis needs for **street views and photos**.
> ⚠️ It was trained on eye-level photos only. On **satellite** images it reliably separates **tree canopy** from everything else, but roads, parking lots and roofs all come out as "building" and water often as "sky". For satellite images we therefore also compute simple **pixel-colour indicators** (share of green, grey and blue pixels) in 2.8.

> *Google logos:* satellite and street view images carry a Google logo / copyright line at the bottom. We keep the files as they are, but cut off the bottom 24 pixels **in memory** before computing features, so the logo does not become a "feature".

*If Colab disconnected:* run the cells of Section 0, then continue here — Section 2 reads everything it needs from Drive.

In [ ]:
# 2.0 The image index: every downloaded image with its source, city and location (read from the manifests in Drive)
def build_image_index():
    parts = []
    if ARCH_MANIFEST.exists():
        a = pd.read_csv(ARCH_MANIFEST).reindex(columns=["image_id", "city", "project", "relpath", "lat", "lon"])
        parts.append(pd.DataFrame(dict(image_id=a.image_id, source="archdaily", city=a.city, unit=a.project,
                                       lat=a.lat, lon=a.lon, relpath=a.relpath)))
    for src in ("satellite", "streetview"):
        p = SOURCE_DIRS[src] / f"{src}_manifest.csv"
        if p.exists():
            g = pd.read_csv(p)
            parts.append(pd.DataFrame(dict(image_id=g.image_id, source=src, city=g.city, unit=g.unit,
                                           lat=g.lat, lon=g.lon, relpath=g.relpath)))
    if not parts:
        return pd.DataFrame(columns=["image_id", "source", "city", "unit", "lat", "lon", "relpath"])
    index = pd.concat(parts, ignore_index=True).drop_duplicates(["source", "image_id"])
    on_disk = np.array([(IMG_DIR / str(p)).exists() for p in index.relpath], dtype=bool)
    return index.loc[on_disk].reset_index(drop=True)


INDEX = build_image_index()
if INDEX.empty:
    raise RuntimeError("No images found. Run Section 1 first (or set USE_SHARED_DATA = True).")
display(INDEX.pivot_table(index="city", columns="source", values="image_id", aggfunc="count", fill_value=0))

In [ ]:
# 2.1 Load the two neural networks (downloaded once from the internet, ~550 MB for VGG16 + 15 MB for SegFormer)
#     (a warning "The secret HF_TOKEN does not exist" can be ignored: the model is public)
import cv2
import torch
import torchvision.transforms.functional as TF
from torchvision.models import vgg16, VGG16_Weights
from transformers import SegformerForSemanticSegmentation

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(SEED)
print("Computing on:", DEVICE, "(GPU)" if DEVICE.type == "cuda" else "(no GPU: Runtime > Change runtime type > T4 GPU is ~5x faster)")

# VGG16 convolution layers up to the last ReLU of block 5 (we drop the final max-pool and the classifier)
vgg_conv = vgg16(weights=VGG16_Weights.IMAGENET1K_V1).features[:30].eval().to(DEVICE)

SEG_MODEL_ID = "nvidia/segformer-b0-finetuned-ade-512-512"
seg_model = SegformerForSemanticSegmentation.from_pretrained(SEG_MODEL_ID).eval().to(DEVICE)
ADE_LABELS = [seg_model.config.id2label[i].strip() for i in range(seg_model.config.num_labels)]

IMAGENET_MEAN, IMAGENET_STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]   # both networks expect this normalisation


GOOGLE_LOGO_PX = 24     # bottom strip with the Google logo / copyright text, cut off before computing features


def prepare_image(path, source):
    """The image as the feature extractors see it: RGB, without the Google logo strip."""
    img = load_rgb(path)
    if source in ("satellite", "streetview"):
        img = img.crop((0, 0, img.width, img.height - GOOGLE_LOGO_PX))
    return img


def to_batch(imgs, size):
    """List of PIL images -> normalised tensor (B, 3, size, size). The whole image is resized (not cropped)."""
    return torch.stack([TF.normalize(TF.to_tensor(im.resize((size, size), Image.BILINEAR)), IMAGENET_MEAN, IMAGENET_STD)
                        for im in imgs]).to(DEVICE)


print(f"VGG16: {sum(p.numel() for p in vgg_conv.parameters()) / 1e6:.1f} M parameters | "
      f"SegFormer-b0: {sum(p.numel() for p in seg_model.parameters()) / 1e6:.1f} M parameters, {len(ADE_LABELS)} classes")

# a few example images used in the visual checks below
EXAMPLES = {src: [IMG_DIR / p for p in g.sample(min(2, len(g)), random_state=SEED).relpath]
            for src, g in INDEX.groupby("source")}
EXAMPLE_IMAGES = {src: [prepare_image(p, src) for p in paths] for src, paths in EXAMPLES.items()}

## 2.2 Lens 1 — VGG16 features
The image is resized to 224 × 224 and passed through the convolution layers. The last block produces **512 feature maps** of 14 × 14 cells; each map lights up where a learned pattern (a texture, a window grid, foliage…) appears.
We keep **one number per map: its average** (*global average pooling*) → 512 numbers. Week 4 flattened the maps instead (7 × 7 × 512 = 25,088 numbers) — far longer, and very sensitive to *where* things are in the frame.

In [ ]:
# 2.2 VGG16 lens
VGG_SIZE = 224


@torch.inference_mode()
def vgg_maps(imgs):
    return vgg_conv(to_batch(imgs, VGG_SIZE))                       # (B, 512, 14, 14)


def vgg_features(imgs):
    return vgg_maps(imgs).mean(dim=(2, 3)).float().cpu().numpy()    # (B, 512)


def show_vgg_lens(img):
    fmap = vgg_maps([img])[0].float().cpu().numpy()
    vec = fmap.mean(axis=(1, 2))
    top = np.argsort(vec)[::-1][:3]
    small = np.asarray(img.resize((VGG_SIZE, VGG_SIZE)))
    fig, axes = plt.subplots(1, 6, figsize=(21, 3.3), gridspec_kw={"width_ratios": [1, 1, 1, 1, 1, 2.3]})
    axes[0].imshow(small); axes[0].set_title("input 224×224")
    overlays = [("mean of 512 maps", fmap.mean(0))] + [(f"map #{c} (strong)", fmap[c]) for c in top]
    for ax, (title, a) in zip(axes[1:5], overlays):
        ax.imshow(small)
        ax.imshow(cv2.resize(a, (VGG_SIZE, VGG_SIZE)), cmap="jet", alpha=0.5)
        ax.set_title(title)
    for ax in axes[:5]:
        ax.axis("off")
    axes[5].bar(range(512), vec, width=1.0, color="#4c72b0")
    axes[5].set_title("the 512-number VGG vector of this image"); axes[5].set_xlabel("feature map")
    plt.tight_layout(); plt.show()


for src, imgs in EXAMPLE_IMAGES.items():
    print(f"--- {src} ---")
    show_vgg_lens(imgs[0])

## 2.3 Lens 2 — semantic segmentation (SegFormer, ADE20K classes)
Every pixel gets a class. The feature vector is simply the **share of the image covered by each of the 150 classes**.
We also group classes into a few readable indicators — e.g. *vegetation* = tree + grass + plant + palm… (the "green view index" of urban studies), *sky* (a proxy for openness), *building*.

In [ ]:
# 2.3 Segmentation lens
SEG_SIZE = 512
SEG_GROUPS = {
    "sky": ["sky"],
    "building": ["building", "house", "skyscraper", "tower", "hovel", "windowpane", "door", "column", "awning", "canopy"],
    "wall": ["wall"],                       # facades outdoors, but mostly interior walls in ArchDaily photos
    "vegetation": ["tree", "grass", "plant", "palm", "flower", "field"],
    "road": ["road", "path", "runway", "dirt track"],
    "sidewalk": ["sidewalk", "stairs", "step", "stairway"],
    "vehicle": ["car", "bus", "truck", "van", "minibike", "bicycle", "boat", "ship", "airplane"],
    "person": ["person"],
    "water": ["water", "sea", "river", "lake", "swimming pool", "fountain"],
    "ground": ["earth", "sand", "land", "rock", "hill", "mountain"],
    "street_furniture": ["pole", "streetlight", "signboard", "traffic light", "fence", "railing", "bench", "ashcan"],
    "interior": ["floor", "ceiling", "table", "chair", "sofa", "bed", "cabinet", "lamp", "rug", "painting", "shelf",
                 "curtain", "armchair", "desk", "cushion", "stool", "kitchen island", "countertop", "chandelier"],
}
SEG_GROUP_IDX = {g: [ADE_LABELS.index(n) for n in names if n in ADE_LABELS] for g, names in SEG_GROUPS.items()}
PALETTE = np.random.default_rng(3).integers(40, 255, size=(len(ADE_LABELS), 3)).astype(np.uint8)


@torch.inference_mode()
def seg_labelmaps(imgs):
    logits = seg_model(pixel_values=to_batch(imgs, SEG_SIZE)).logits      # (B, 150, 128, 128)
    return logits.argmax(1).cpu().numpy()


def seg_features(labelmaps):
    n = len(ADE_LABELS)
    return np.stack([np.bincount(lm.ravel(), minlength=n) / lm.size for lm in labelmaps]).astype(np.float32)


def show_seg_lens(img):
    lm = seg_labelmaps([img])[0]
    frac = seg_features([lm])[0]
    mask = cv2.resize(PALETTE[lm], img.size, interpolation=cv2.INTER_NEAREST)
    groups = {g: frac[idx].sum() for g, idx in SEG_GROUP_IDX.items()}
    fig, axes = plt.subplots(1, 3, figsize=(18, 3.8), gridspec_kw={"width_ratios": [1.2, 1.2, 1.3]})
    axes[0].imshow(img); axes[0].set_title("image"); axes[0].axis("off")
    axes[1].imshow(img); axes[1].imshow(mask, alpha=0.6); axes[1].set_title("SegFormer classes"); axes[1].axis("off")
    top = [i for i in np.argsort(frac)[::-1][:8] if frac[i] > 0]
    axes[1].legend([plt.Rectangle((0, 0), 1, 1, color=PALETTE[i] / 255) for i in top],
                   [f"{ADE_LABELS[i]} {frac[i]:.0%}" for i in top], loc="center left", bbox_to_anchor=(1.0, 0.5), fontsize=8)
    axes[2].barh(list(groups)[::-1], list(groups.values())[::-1], color="#55a868")
    axes[2].set_xlim(0, 1); axes[2].set_title("grouped indicators (share of pixels)")
    plt.tight_layout(); plt.show()


for src, imgs in EXAMPLE_IMAGES.items():
    print(f"--- {src} ---")
    for img in imgs:
        show_seg_lens(img)

## 2.4 Lens 3 — edges and their orientation
Canny (Week 4) marks strong intensity changes. Instead of keeping all edge pixels we summarise them:
* **edge density on an 8 × 8 grid** (64 numbers) — where the image is busy;
* **orientation histogram** (16 bins) — the gradient direction of each edge pixel, weighted by how strong the edge is. A *vertical* line (facade edge, pole) has a *horizontal* gradient (0°/180°); a *horizontal* line (cornice, kerb, horizon) has a 90° gradient;
* **overall edge density** (1 number).

The thresholds adapt to each image's brightness (the median rule), so dark and bright images are treated alike.

In [ ]:
# 2.4 Edge lens
EDGE_SIZE, EDGE_GRID, EDGE_BINS = 256, 8, 16
EDGE_DIM = EDGE_GRID * EDGE_GRID + EDGE_BINS + 1
VERTICAL_BINS = [0, 1, EDGE_BINS - 2, EDGE_BINS - 1]        # gradient within 22.5° of horizontal -> vertical lines
HORIZONTAL_BINS = list(range(EDGE_BINS // 2 - 2, EDGE_BINS // 2 + 2))   # gradient near 90° -> horizontal lines


def edge_analysis(img):
    gray = cv2.cvtColor(np.asarray(img.resize((EDGE_SIZE, EDGE_SIZE))), cv2.COLOR_RGB2GRAY)
    gray = cv2.GaussianBlur(gray, (3, 3), 0)
    med = float(np.median(gray))
    lo = int(max(10, 0.66 * med))
    hi = int(min(255, max(lo + 20, 1.33 * med)))
    edges = cv2.Canny(gray, lo, hi) > 0
    gx = cv2.Sobel(gray, cv2.CV_32F, 1, 0, ksize=3)
    gy = cv2.Sobel(gray, cv2.CV_32F, 0, 1, ksize=3)
    angle = (np.degrees(np.arctan2(-gy, gx)) + 180) % 180         # -gy: image rows go down, angles go up
    hist, _ = np.histogram(angle[edges], bins=EDGE_BINS, range=(0, 180), weights=np.hypot(gx, gy)[edges])
    hist = hist / (hist.sum() + 1e-9)
    c = EDGE_SIZE // EDGE_GRID
    grid = edges.reshape(EDGE_GRID, c, EDGE_GRID, c).mean(axis=(1, 3))
    feat = np.concatenate([grid.ravel(), hist, [edges.mean()]]).astype(np.float32)
    return feat, dict(gray=gray, edges=edges, grid=grid, hist=hist)


def show_edge_lens(img):
    feat, d = edge_analysis(img)
    fig = plt.figure(figsize=(18, 3.6))
    ax = fig.add_subplot(1, 4, 1); ax.imshow(img.resize((EDGE_SIZE, EDGE_SIZE))); ax.set_title("image"); ax.axis("off")
    ax = fig.add_subplot(1, 4, 2); ax.imshow(d["edges"], cmap="gray"); ax.set_title(f"Canny edges (density {feat[-1]:.2f})"); ax.axis("off")
    ax = fig.add_subplot(1, 4, 3); im = ax.imshow(d["grid"], cmap="magma"); ax.set_title("edge density, 8×8 grid"); ax.axis("off")
    plt.colorbar(im, ax=ax, fraction=0.046)
    ax = fig.add_subplot(1, 4, 4, projection="polar")
    width = np.pi / EDGE_BINS
    centers = np.deg2rad(np.arange(EDGE_BINS) * 180 / EDGE_BINS) + width / 2 + np.pi / 2   # rotate: gradient -> line direction
    ax.bar(np.concatenate([centers, centers + np.pi]), np.concatenate([d["hist"], d["hist"]]), width=width, color="#c44e52")
    ax.set_yticklabels([]); ax.set_title("direction of the edge LINES\n(vertical lines point up)")
    ax.set_theta_zero_location("E")
    plt.tight_layout(); plt.show()
    print(f"vertical-line share {d['hist'][VERTICAL_BINS].sum():.2f} | horizontal-line share {d['hist'][HORIZONTAL_BINS].sum():.2f}")


for src, imgs in EXAMPLE_IMAGES.items():
    print(f"--- {src} ---")
    show_edge_lens(imgs[0])

## 2.5 Lens 4 — colour
* **HSV histograms**: hue (18 bins, only for clearly coloured pixels), saturation (8), value/brightness (8);
* **mean and standard deviation of R, G, B** and a **colourfulness** score (Hasler & Süsstrunk, 2003);
* **4 × 4 grid of average colours** (48 numbers) — the colour *layout* (sky on top, trees at the side, asphalt below).

The dominant colours shown on the right (k-means on the pixels) are only for visualisation.

In [ ]:
# 2.5 Colour lens
from sklearn.cluster import KMeans

COLOR_SIZE = 128
COLOR_DIM = 18 + 8 + 8 + 6 + 1 + 48


def color_analysis(img):
    im = np.asarray(img.resize((COLOR_SIZE, COLOR_SIZE)))
    hsv = cv2.cvtColor(im, cv2.COLOR_RGB2HSV)                       # H: 0-179, S and V: 0-255
    n = COLOR_SIZE * COLOR_SIZE
    coloured = (hsv[..., 1] > 40) & (hsv[..., 2] > 40)              # hue is meaningless for grey or dark pixels
    h_hist = np.histogram(hsv[..., 0][coloured], bins=18, range=(0, 180))[0] / n
    s_hist = np.histogram(hsv[..., 1], bins=8, range=(0, 256))[0] / n
    v_hist = np.histogram(hsv[..., 2], bins=8, range=(0, 256))[0] / n
    rgb = im.reshape(-1, 3).astype(np.float32) / 255
    rg = im[..., 0].astype(np.float32) - im[..., 1]
    yb = 0.5 * (im[..., 0].astype(np.float32) + im[..., 1]) - im[..., 2]
    colourfulness = (np.hypot(rg.std(), yb.std()) + 0.3 * np.hypot(rg.mean(), yb.mean())) / 100
    layout = im.reshape(4, COLOR_SIZE // 4, 4, COLOR_SIZE // 4, 3).mean(axis=(1, 3)) / 255
    feat = np.concatenate([h_hist, s_hist, v_hist, rgb.mean(0), rgb.std(0), [colourfulness], layout.ravel()])
    return feat.astype(np.float32), dict(h_hist=h_hist, layout=layout, colourfulness=colourfulness)


def show_color_lens(img):
    feat, d = color_analysis(img)
    px = np.asarray(img.resize((64, 64))).reshape(-1, 3).astype(np.float32)
    km = KMeans(n_clusters=5, n_init=3, random_state=SEED).fit(px)
    share = np.bincount(km.labels_, minlength=5) / len(px)
    order = np.argsort(share)[::-1]
    fig, axes = plt.subplots(1, 4, figsize=(18, 3.3), gridspec_kw={"width_ratios": [1, 1.6, 0.8, 1.6]})
    axes[0].imshow(img); axes[0].set_title("image"); axes[0].axis("off")
    hues = mpl.colors.hsv_to_rgb(np.column_stack([(np.arange(18) + 0.5) / 18, np.ones(18), np.ones(18)]))
    axes[1].bar(range(18), d["h_hist"], color=hues); axes[1].set_title("hue histogram (coloured pixels only)")
    axes[1].set_xticks([])
    axes[2].imshow(d["layout"]); axes[2].set_title("4×4 colour layout"); axes[2].axis("off")
    left = 0.0
    for k in order:
        axes[3].barh(0, share[k], left=left, color=km.cluster_centers_[k] / 255)
        left += share[k]
    axes[3].set_xlim(0, 1); axes[3].set_yticks([])
    axes[3].set_title(f"dominant colours · colourfulness {d['colourfulness']:.2f}")
    plt.tight_layout(); plt.show()


for src, imgs in EXAMPLE_IMAGES.items():
    print(f"--- {src} ---")
    show_color_lens(imgs[0])

## 2.6 Extract the four lenses for every image
Images are processed in batches (on the GPU if available). Results are saved **per source and per city** in `02_features/raw/<source>/<city>.npz`.
If a file already contains exactly the images on disk, computed with the same settings, it is skipped — so after a disconnection you only compute what is missing.
Each file holds the **long vector** of every image (`long_vector`, 832 numbers), the same numbers split by lens (`vgg`, `seg`, `edge`, `color`), the image ids and the **name of every column** (`feature_names`).
On a T4 GPU this takes about 3–5 minutes for ~2,000 images (most of the time is reading the images from Drive).

In [ ]:
# 2.6 Batch extraction -> raw feature files
BATCH_SIZE = 16
RECOMPUTE_FEATURES = False
BLOCKS = ["vgg", "seg", "edge", "color"]
BLOCK_DIMS = {"vgg": 512, "seg": len(ADE_LABELS), "edge": EDGE_DIM, "color": COLOR_DIM}


FEATURE_SETTINGS = json.dumps(dict(VGG_SIZE=VGG_SIZE, SEG_MODEL_ID=SEG_MODEL_ID, SEG_SIZE=SEG_SIZE, EDGE_SIZE=EDGE_SIZE,
                                    EDGE_GRID=EDGE_GRID, EDGE_BINS=EDGE_BINS, COLOR_SIZE=COLOR_SIZE,
                                    GOOGLE_LOGO_PX=GOOGLE_LOGO_PX), sort_keys=True)


def feature_names():
    names = [f"vgg_{i:03d}" for i in range(512)] + [f"seg_{slugify(n)}" for n in ADE_LABELS]
    names += [f"edge_density_r{r}c{c}" for r in range(EDGE_GRID) for c in range(EDGE_GRID)]
    names += [f"edge_orient_{int(a)}deg" for a in np.arange(EDGE_BINS) * 180 / EDGE_BINS] + ["edge_density_all"]
    names += [f"hue_{i:02d}" for i in range(18)] + [f"sat_{i}" for i in range(8)] + [f"val_{i}" for i in range(8)]
    names += ["mean_r", "mean_g", "mean_b", "std_r", "std_g", "std_b", "colourfulness"]
    names += [f"layout_r{r}c{c}_{ch}" for r in range(4) for c in range(4) for ch in "rgb"]
    return names


def extract_blocks(paths, source, desc=""):
    out, ok = {b: [] for b in BLOCKS}, []
    for start in tqdm(range(0, len(paths), BATCH_SIZE), desc=desc, leave=False):
        imgs = []
        for i, p in enumerate(paths[start:start + BATCH_SIZE]):
            try:
                imgs.append(prepare_image(p, source))
                ok.append(start + i)
            except Exception as e:
                print(f"  cannot read {p}: {e}")
        if not imgs:
            continue
        out["vgg"].append(vgg_features(imgs))
        out["seg"].append(seg_features(seg_labelmaps(imgs)))
        out["edge"].append(np.stack([edge_analysis(im)[0] for im in imgs]))
        out["color"].append(np.stack([color_analysis(im)[0] for im in imgs]))
    blocks = {b: (np.concatenate(v) if v else np.zeros((0, BLOCK_DIMS[b]))).astype(np.float32) for b, v in out.items()}
    return ok, blocks


t0 = time.time()
for (src, city), g in INDEX.groupby(["source", "city"], sort=False):
    f = RAW_DIR / src / f"{city}.npz"
    if f.exists() and not RECOMPUTE_FEATURES:
        with np.load(f) as z:
            same_images = set(z["image_id"].astype(str)) == set(g.image_id.astype(str))
            same_settings = "settings" in z.files and str(z["settings"]) == FEATURE_SETTINGS
        if same_images and same_settings:
            print(f"{src:10s} {city:13s} {len(g):4d} images  (already extracted)")
            continue
    ok, blocks = extract_blocks([IMG_DIR / p for p in g.relpath], src, desc=f"{src}/{city}")
    np.savez_compressed(f, image_id=g.image_id.to_numpy().astype(str)[ok], long_vector=np.hstack([blocks[b] for b in BLOCKS]),
                        feature_names=np.array(feature_names()), settings=np.array(FEATURE_SETTINGS), **blocks)
    print(f"{src:10s} {city:13s} {len(ok):4d} images  -> {f.name} ({f.stat().st_size / 1e6:.1f} MB)")
print(f"done in {time.time() - t0:.0f} s")

In [ ]:
# 2.7 Load the raw features back (one table + four matrices per source)
def load_raw(source):
    known = INDEX[INDEX.source == source].set_index("image_id")
    metas, parts = [], {b: [] for b in BLOCKS}
    for f in sorted((RAW_DIR / source).glob("*.npz")):
        with np.load(f) as z:
            ids = z["image_id"].astype(str)
            keep = np.isin(ids, known.index.astype(str))
            if not keep.any():
                continue
            for b in BLOCKS:
                parts[b].append(z[b][keep])
        metas.append(known.loc[ids[keep]].reset_index())
    if not metas:
        return None, None
    return pd.concat(metas, ignore_index=True), {b: np.concatenate(v) for b, v in parts.items()}


# how to read one raw file yourself:
#   z = np.load(RAW_DIR / "streetview" / "Miami.npz");  z["long_vector"].shape, z["feature_names"][:5], z["image_id"][:3]
RAW = {}
for src in SOURCES:
    meta, blocks = load_raw(src)
    if meta is None:
        print(f"{src}: no features")
        continue
    RAW[src] = (meta, blocks)
    size_mb = sum(f.stat().st_size for f in (RAW_DIR / src).glob("*.npz")) / 1e6
    dims = " + ".join(f"{b} {blocks[b].shape[1]}" for b in BLOCKS)
    print(f"{src:10s} {len(meta):5d} images × ({dims} = {sum(blocks[b].shape[1] for b in BLOCKS)} numbers)  "
          f"-> {size_mb:.1f} MB as .npz")

## 2.8 Readable indicators: do the cities differ?
Some features are directly interpretable. We compute a small table of **indicators** per image (prefix `int_`): shares of vegetation, sky, building… from the segmentation, edge density and the vertical / horizontal line shares from the edges, brightness and colourfulness from the colour lens.
For satellite images we add three **pixel-colour indicators** that do not depend on SegFormer: the share of clearly green pixels (vegetation, lawns), of grey/unsaturated pixels (asphalt, concrete, many roofs) and of blue pixels (water, pools).
These indicators are **not** used for clustering (the embedding is); we keep them in the CSVs to **interpret** the clusters later.

In [ ]:
# 2.8 Interpretable indicators + comparison between cities  (INT_COLS_BY_SOURCE, the columns plotted per source, is set in 0.4)


def interpretable_table(blocks):
    seg, edge, color = blocks["seg"], blocks["edge"], blocks["color"]
    t = {f"int_{g}": seg[:, idx].sum(axis=1) for g, idx in SEG_GROUP_IDX.items()}
    hist = edge[:, EDGE_GRID * EDGE_GRID: EDGE_GRID * EDGE_GRID + EDGE_BINS]
    t["int_edge_density"] = edge[:, -1]
    t["int_vertical_edges"] = hist[:, VERTICAL_BINS].sum(axis=1)
    t["int_horizontal_edges"] = hist[:, HORIZONTAL_BINS].sum(axis=1)
    t["int_brightness"] = color[:, 34:37].mean(axis=1)          # mean of R, G, B
    t["int_colourfulness"] = color[:, 40]
    hue = color[:, 0:18]                                        # share of all pixels, per 10-unit hue bin (OpenCV hue 0-179)
    t["int_green_pixels"] = hue[:, 3:9].sum(axis=1)              # hue 30-90 = yellow-green to green
    t["int_blue_pixels"] = hue[:, 9:13].sum(axis=1)              # hue 90-130 = cyan to blue
    t["int_grey_pixels"] = color[:, 18:20].sum(axis=1)           # saturation < 64 = grey, white, black
    return pd.DataFrame(t)


INTERP = {src: interpretable_table(blocks) for src, (meta, blocks) in RAW.items()}


def plot_city_comparison(src):
    meta = RAW[src][0]
    t = INTERP[src].assign(city=meta.city.values)
    cities = [c for c in CITY_ORDER if c in set(t.city)]
    cols = INT_COLS_BY_SOURCE[src]
    fig, axes = plt.subplots(1, len(cols), figsize=(2.6 * len(cols), 3.4), sharey=False, squeeze=False)
    axes = axes[0]
    for ax, col in zip(axes, cols):
        bp = ax.boxplot([t.loc[t.city == c, col].values for c in cities], patch_artist=True, showfliers=False, widths=0.6)
        for patch, c in zip(bp["boxes"], cities):
            patch.set_facecolor(city_color(c))
        ax.set_xticks(range(1, len(cities) + 1))
        ax.set_xticklabels([c[:4] for c in cities], rotation=90, fontsize=8)
        ax.set_title(col.replace("int_", ""), fontsize=10)
    fig.suptitle(f"{src}: indicators per city", fontsize=12)
    plt.tight_layout()
    fig.savefig(FEAT_FIG_DIR / f"{src}_city_indicators.png", dpi=110, bbox_inches="tight")
    plt.show()


for src in RAW:
    plot_city_comparison(src)

✏️ **Discuss:** Which city has the greenest streets? The most sky? Do the satellite indicators agree with the street-level ones (e.g. vegetation from above vs from the street)?

## 2.9 From four lenses to one embedding

**Problem 1 – balance.** VGG has 512 columns, segmentation 150, edges 81, colour 89, and their values live on different scales (VGG averages are unbounded, shares are between 0 and 1). If we simply concatenate, VGG dominates every distance just because it has the most columns.
**Problem 2 – length.** 832 numbers per image, many redundant: SOM training is slower and distances become less meaningful ("curse of dimensionality").

Our fusion recipe (an sklearn `Pipeline`, saved to Drive so new images can be processed the same way later):
1. **square root** of the non-negative blocks (segmentation, edges, colour) — for the histograms this is the Hellinger transform, which stops a few dominant classes from overwhelming rare ones;
2. **standardise** every column (mean 0, std 1), clip extreme values to ±5, and standardise again (clipping changes the spread);
3. drop segmentation classes that almost never occur (present in < 2 % of the images);
4. **weight each block by 1/√(number of columns)** → every lens contributes the same total variance to the distances between images (change `BLOCK_WEIGHTS` to favour one). PCA then keeps the directions with most variance, so the balance inside the embedding is only approximately equal;
5. **PCA** → keep the first *k* components = the embedding.

The pipeline is fitted **per source on all cities together** (one shared coordinate system, so cities can be compared and one SOM can be trained per source). That also matters for the size of the embedding: PCA can give at most *n − 1* meaningful components for *n* images — with 150 images per city a per-city PCA could never give 150 dimensions.

In [ ]:
# 2.9a Fusion pipeline (built from standard sklearn parts, so it can be saved and re-loaded anywhere)
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import StandardScaler, FunctionTransformer
from sklearn.decomposition import PCA
from sklearn.manifold import trustworthiness, TSNE
from sklearn.neighbors import NearestNeighbors
import joblib

BLOCK_WEIGHTS = {"vgg": 1.0, "seg": 1.0, "edge": 1.0, "color": 1.0}
SQRT_BLOCKS = {"seg", "edge", "color"}
CLIP_Z = 5.0
SEG_MIN_PRESENCE = 0.02      # keep a segmentation class if >0.5% of the pixels in at least 2% of the images


def stack_blocks(blocks):
    return np.hstack([blocks[b] for b in BLOCKS]).astype(np.float64)


def make_fusion(blocks, weights=BLOCK_WEIGHTS):
    offsets = np.cumsum([0] + [blocks[b].shape[1] for b in BLOCKS])
    transformers, block_weight, kept = [], {}, {}
    for i, b in enumerate(BLOCKS):
        cols = np.arange(offsets[i], offsets[i + 1])
        if b == "seg":
            present = (blocks["seg"] > 0.005).mean(axis=0) >= SEG_MIN_PRESENCE
            cols = cols[present]
            kept["seg_classes"] = [ADE_LABELS[j] for j in np.where(present)[0]]
        steps = ([FunctionTransformer(np.sqrt)] if b in SQRT_BLOCKS else []) + [
            StandardScaler(), FunctionTransformer(np.clip, kw_args={"a_min": -CLIP_Z, "a_max": CLIP_Z}), StandardScaler()]
        transformers.append((b, make_pipeline(*steps), cols.tolist()))
        n_varying = int((blocks[b][:, cols - offsets[i]].std(axis=0) > 1e-9).sum())
        block_weight[b] = weights[b] / math.sqrt(max(n_varying, 1))
        kept[b] = n_varying
    return ColumnTransformer(transformers, transformer_weights=block_weight, remainder="drop"), kept


FUSED = {}
for src, (meta, blocks) in RAW.items():
    if len(meta) < 3:
        print(f"{src}: only {len(meta)} images -> skipped")
        continue
    X_raw = stack_blocks(blocks)
    fusion, kept = make_fusion(blocks)
    X_f = fusion.fit_transform(X_raw)
    FUSED[src] = dict(meta=meta, X_raw=X_raw, fusion=fusion, X_f=X_f, kept=kept)
    print(f"{src:10s} raw {X_raw.shape[1]} columns -> {X_f.shape[1]} after dropping rare classes | "
          f"varying columns per block: { {b: kept[b] for b in BLOCKS} }")

### 2.9b How many dimensions do we need?
We compare embeddings of different size *k* with three measures:
* **explained variance** — how much of the total spread of the fused data the first *k* PCA components keep;
* **trustworthiness** (0–1) — are the nearest neighbours in the embedding also near neighbours in the full data? (1 = no "false" neighbours);
* **k-NN overlap** — share of each image's 10 nearest neighbours (fewer for very small data sets) that stay the same.

The curves flatten: after some *k*, extra dimensions add little. `N_COMPONENTS = "auto"` picks the smallest *k* that keeps `VARIANCE_TARGET` of the variance (at most 100, and at most n − 1). You can also force a number, e.g. `N_COMPONENTS = 50`, and compare the SOM results.

In [ ]:
# 2.9b Explore the embedding size
N_COMPONENTS = "auto"     # or an integer, e.g. 50
VARIANCE_TARGET = 0.90
MAX_COMPONENTS = 100
K_GRID = [2, 5, 10, 20, 30, 50, 75, 100, 150, 200]


def explore_dimensions(X_f, n_neighbors=10):
    n = len(X_f)
    nn = max(1, min(n_neighbors, n // 2 - 1))
    pca = PCA(svd_solver="full").fit(X_f)
    Z = pca.transform(X_f)
    cum = np.cumsum(pca.explained_variance_ratio_)
    ref = NearestNeighbors(n_neighbors=nn + 1).fit(X_f).kneighbors(X_f, return_distance=False)[:, 1:]
    rows = []
    for k in [k for k in K_GRID if k < min(n, X_f.shape[1])]:
        Zk = Z[:, :k]
        nbr = NearestNeighbors(n_neighbors=nn + 1).fit(Zk).kneighbors(Zk, return_distance=False)[:, 1:]
        overlap = np.mean([len(set(a) & set(b)) / nn for a, b in zip(ref, nbr)])
        rows.append(dict(k=k, explained_variance=cum[k - 1], trustworthiness=trustworthiness(X_f, Zk, n_neighbors=nn),
                         knn_overlap=overlap))
    return cum, pd.DataFrame(rows), nn


def choose_k(cum, n):
    if N_COMPONENTS != "auto":
        return int(min(N_COMPONENTS, n - 1, len(cum)))
    k = int(np.searchsorted(cum, VARIANCE_TARGET) + 1)
    return int(np.clip(k, 2, min(MAX_COMPONENTS, n - 1, len(cum))))


fig, axes = plt.subplots(1, len(FUSED), figsize=(6 * len(FUSED), 3.8), squeeze=False)
for ax, (src, d) in zip(axes[0], FUSED.items()):
    cum, table, nn = explore_dimensions(d["X_f"])
    d["cum_var"], d["dim_table"], d["k"] = cum, table, choose_k(cum, len(d["X_f"]))
    ax.plot(np.arange(1, len(cum) + 1), cum, color="#999", lw=1, label="explained variance (all k)")
    ax.plot(table.k, table.explained_variance, "o", color="#999")
    ax.plot(table.k, table.trustworthiness, "s-", color="#4c72b0", label="trustworthiness")
    ax.plot(table.k, table.knn_overlap, "^-", color="#dd8452", label=f"{nn}-NN overlap")
    ax.axvline(d["k"], color="crimson", ls="--", label=f"chosen k = {d['k']}")
    ax.set_xscale("log"); ax.set_ylim(0, 1.02); ax.set_xlabel("embedding size k"); ax.set_title(f"{src} (n = {len(d['X_f'])})")
    ax.legend(fontsize=8, loc="lower right")
plt.tight_layout()
fig.savefig(FEAT_FIG_DIR / "embedding_size.png", dpi=110, bbox_inches="tight")
plt.show()
for src, d in FUSED.items():
    print(f"--- {src}: chosen k = {d['k']} (keeps {d['cum_var'][d['k'] - 1]:.0%} of the variance)")
    display(d["dim_table"].round(3).set_index("k").T)

In [ ]:
# 2.9c Final pipeline per source: fusion + PCA(k). Saved to Drive for later re-use.
for src, d in FUSED.items():
    pipe = Pipeline([("fusion", d["fusion"]), ("pca", PCA(n_components=d["k"], svd_solver="full"))])
    d["Z"] = pipe.fit_transform(d["X_raw"])
    d["pipeline"] = pipe
    joblib.dump(pipe, FEAT_MODEL_DIR / f"{src}_fusion_pca.joblib")
    info = dict(source=src, n_images=len(d["Z"]), k=d["k"], explained_variance=float(d["cum_var"][d["k"] - 1]),
                blocks=BLOCKS, block_dims={b: int(RAW[src][1][b].shape[1]) for b in BLOCKS},
                block_weights=BLOCK_WEIGHTS, seg_classes_used=d["kept"]["seg_classes"],
                created=datetime.now().isoformat(timespec="seconds"))
    (FEAT_MODEL_DIR / f"{src}_fusion_pca.json").write_text(json.dumps(info, indent=1))
    print(f"{src:10s} embedding {d['Z'].shape} saved model -> {src}_fusion_pca.joblib")

## 2.10 Look at the embedding
**(a) Same query, different lenses.** For one image we search its 5 most similar images using each lens alone, and using everything together. Each lens has its own idea of "similar"!

In [ ]:
# 2.10a Nearest neighbours under each lens
QUERY_SOURCE = "streetview" if "streetview" in FUSED else next(iter(FUSED))
QUERY_POSITION = 0        # change me: any number between 0 and the number of images - 1


def show_neighbours(src, q, k=5):
    d = FUSED[src]
    meta, X_f, fusion = d["meta"], d["X_f"], d["fusion"]
    spaces = {f"{b} only": X_f[:, fusion.output_indices_[b]] for b in BLOCKS}
    spaces["all lenses (fused)"] = X_f
    spaces[f"embedding (k={d['k']})"] = d["Z"]
    fig, axes = plt.subplots(len(spaces), k + 1, figsize=(2.2 * (k + 1), 2.1 * len(spaces)))
    for row, (name, S) in enumerate(spaces.items()):
        S = S / (np.linalg.norm(S, axis=1, keepdims=True) + 1e-9)
        sims = S @ S[q]
        sims[q] = -np.inf
        best = [i for i in np.argsort(sims)[::-1][:k] if i != q]
        for col, i in enumerate([q] + list(best)):
            ax = axes[row, col]
            im = load_rgb(IMG_DIR / meta.relpath.iloc[i]); im.thumbnail((256, 256))
            ax.imshow(im); ax.set_xticks([]); ax.set_yticks([])
            ax.set_title("QUERY" if col == 0 else f"{meta.city.iloc[i][:10]} {sims[i]:.2f}", fontsize=8)
        axes[row, 0].set_ylabel(name, fontsize=9)
    plt.tight_layout(); plt.show()


show_neighbours(QUERY_SOURCE, QUERY_POSITION % len(FUSED[QUERY_SOURCE]["meta"]))

**(b) What do the main axes mean?** Each image is placed at its (PC1, PC2) coordinates. Then we correlate every principal component with the readable indicators.

In [ ]:
# 2.10b Images on the first two principal components + what the components correlate with
from matplotlib.offsetbox import OffsetImage, AnnotationBbox

EXPLORE_SOURCE = QUERY_SOURCE


def pc_image_scatter(src, n=120, thumb=56):
    d = FUSED[src]
    Z, meta = d["Z"], d["meta"]
    rng = np.random.default_rng(SEED)
    pick = rng.choice(len(Z), size=min(n, len(Z)), replace=False)
    fig, ax = plt.subplots(figsize=(12, 9))
    ax.scatter(Z[:, 0], Z[:, 1], s=4, c="#ccc")
    for i in pick:
        im = load_rgb(IMG_DIR / meta.relpath.iloc[i]); im.thumbnail((thumb, thumb))
        ax.add_artist(AnnotationBbox(OffsetImage(np.asarray(im), zoom=1), (Z[i, 0], Z[i, 1]), frameon=True, pad=0.1,
                                     bboxprops=dict(edgecolor=city_color(meta.city.iloc[i]), lw=1.5)))
    ax.set_xlabel("PC1"); ax.set_ylabel("PC2"); ax.set_title(f"{src}: images placed by the first two components (frame = city)")
    plt.tight_layout()
    fig.savefig(FEAT_FIG_DIR / f"{src}_pc_scatter.png", dpi=110, bbox_inches="tight")
    plt.show()


def pc_correlations(src, n_pc=4):
    Z, t = FUSED[src]["Z"], INTERP[src]
    cols = [c for c in t.columns if t[c].std() > 0]
    n_pc = min(n_pc, Z.shape[1])
    corr = pd.DataFrame({f"PC{j + 1}": [np.corrcoef(Z[:, j], t[c])[0, 1] for c in cols] for j in range(n_pc)},
                        index=[c.replace("int_", "") for c in cols])
    fig, ax = plt.subplots(figsize=(1.4 * n_pc + 2, 0.32 * len(cols) + 1))
    im = ax.imshow(corr.values, cmap="RdBu_r", vmin=-1, vmax=1, aspect="auto")
    ax.set_xticks(range(n_pc)); ax.set_xticklabels(corr.columns)
    ax.set_yticks(range(len(cols))); ax.set_yticklabels(corr.index)
    for (r, c), v in np.ndenumerate(corr.values):
        ax.text(c, r, f"{v:.2f}", ha="center", va="center", fontsize=7)
    plt.colorbar(im, ax=ax); ax.set_title(f"{src}: correlation of components with indicators")
    plt.tight_layout(); plt.show()


pc_image_scatter(EXPLORE_SOURCE)
pc_correlations(EXPLORE_SOURCE)

In [ ]:
# 2.10c t-SNE map of each source, coloured by city: do cities form their own regions?
fig, axes = plt.subplots(1, len(FUSED), figsize=(6 * len(FUSED), 5), squeeze=False)
for ax, (src, d) in zip(axes[0], FUSED.items()):
    n = len(d["Z"])
    if n < 6:
        ax.set_title(f"{src}: too few images"); continue
    xy = TSNE(n_components=2, perplexity=float(min(30, max(2, (n - 1) // 3))), init="pca",
              random_state=SEED).fit_transform(d["Z"])
    for city in [c for c in CITY_ORDER if c in set(d["meta"].city)]:
        m = (d["meta"].city == city).to_numpy()
        ax.scatter(xy[m, 0], xy[m, 1], s=12, color=city_color(city), label=city, alpha=0.8)
    ax.set_title(f"{src} — t-SNE of the embedding"); ax.set_xticks([]); ax.set_yticks([])
    ax.legend(fontsize=7, markerscale=1.5)
plt.tight_layout()
fig.savefig(FEAT_FIG_DIR / "tsne_by_city.png", dpi=110, bbox_inches="tight")
plt.show()

## 2.11 Save the embeddings: one CSV per source and per city
Each row is one image: identifiers (`image_id`, `source`, `city`, `unit`, `lat`, `lon`, `relpath`), the embedding (`emb_000 … emb_k`) and the readable indicators (`int_…`).
The same `image_id` in `satellite/` and `streetview/` means **the same location**. Section 3 will add the SOM labels to these files.

In [ ]:
# 2.11 Write embeddings/<source>/<city>.csv
ID_COLS = ["image_id", "source", "city", "unit", "lat", "lon", "relpath"]

for src, d in FUSED.items():
    emb_cols = [f"emb_{j:03d}" for j in range(d["Z"].shape[1])]
    table = pd.concat([d["meta"][ID_COLS].reset_index(drop=True),
                       pd.DataFrame(d["Z"], columns=emb_cols),
                       INTERP[src].reset_index(drop=True)], axis=1)
    out_dir = EMB_DIR / src
    for old in out_dir.glob("*.csv"):          # remove files of a previous run (cities may have changed)
        old.unlink()
    for city, g in table.groupby("city", sort=False):
        g.to_csv(out_dir / f"{city}.csv", index=False, float_format="%.8g")
    print(f"{src:10s} {len(table):5d} rows × {table.shape[1]} columns -> {sorted(p.name for p in out_dir.glob('*.csv'))}")

example = next(iter((EMB_DIR / QUERY_SOURCE).glob("*.csv")))
display(pd.read_csv(example).iloc[:3, :12])

---
# 3 · Clustering with Self-Organizing Maps — and checking that the result is reliable

In Weeks 5–6 we trained one SOM with one size and looked at the picture. A different size, or just a different random seed, could have given a different story.
Today we **measure** before we interpret:

| Metric | Question it answers | Better |
|---|---|---|
| **Quantization error (QE)** | how far is each image from its best-matching node? (fit) | lower |
| **Topographic error (TE)** | for how many images are the best and second-best nodes *not* neighbours? (is the map smooth?) | lower |
| **Empty nodes** | share of nodes that no image chose (many = map too big; a few on the borders between groups are normal) | lower |
| **Silhouette** | are the clusters compact and well separated? (−1 … 1) | higher |
| **Davies–Bouldin** | average similarity between each cluster and its closest one | lower |
| **Stability (ARI)** | train again with another random seed: do we get the same grouping? (0 = random, 1 = identical) — for the nodes and for the clusters | higher |

As the map grows, QE falls (more nodes fit the data more closely), but each node holds fewer images, more nodes stay empty and the result depends more on the random start — the best size is a **compromise**.
(TE usually stays close to 0 here: unlike Weeks 5–6, where `sigma = 1` was fixed, the neighbourhood starts wide — half the map — and shrinks to 1 during training, which keeps the map smooth.)

**Two levels of labels.** A SOM node is a small group of very similar images (`som_node`). Neighbouring nodes are then merged into a few **clusters** (`som_cluster`) by hierarchical clustering of the node weights, constrained so that every cluster is a connected region of the map.

The SOM input is the embedding saved in Section 2 (`emb_…` columns of the CSVs). We train **one SOM per source** on all its cities (cities share the same embedding space). You can restrict it to some cities with `CITY_FILTER` — such a SOM is saved under its own name and does not replace the all-cities results.

*If Colab disconnected:* run the cells of Section 0, then continue here (Section 3 reads the CSVs from Drive). Only the demo in 3.6c, which processes a new image, also needs the cells 2.0–2.5 (the feature extractors); without them it is simply skipped.

In [ ]:
# 3.1 Tools: load embeddings, train and evaluate a SOM
from minisom import MiniSom
from sklearn.cluster import AgglomerativeClustering
from sklearn.feature_extraction.image import grid_to_graph
from sklearn.metrics import silhouette_score, davies_bouldin_score, adjusted_rand_score
from importlib.metadata import version as package_version
import sklearn
import joblib

SOM_SEEDS = [0, 1, 2, 3]       # every size is trained 4 times with different random starts (at least 2 are needed)
K_EVAL = 6                     # number of clusters used to compare sizes
SOM_LEARNING_RATE = 0.5
SOM_COLS = ["som_row", "som_col", "som_node", "som_cluster", "som_bmu_dist"]
CLUSTER_COLORS = [mpl.colors.to_hex(c) for c in mpl.colormaps["tab10"].colors]
assert len(SOM_SEEDS) >= 2, "SOM_SEEDS needs at least 2 seeds to measure stability"


def load_embeddings(source, cities=None):
    files = sorted((EMB_DIR / source).glob("*.csv"))
    if not files:
        raise FileNotFoundError(f"No embedding CSVs for {source}: run Section 2 first.")
    df = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)
    if cities:
        if isinstance(cities, str):              # CITY_FILTER = "Miami" is read as ["Miami"]
            cities = [cities]
        available = sorted(set(df.city))
        wanted = {str(c).strip().lower() for c in cities}
        unknown = wanted - {c.lower() for c in available}
        if unknown:
            raise ValueError(f"CITY_FILTER: {sorted(unknown)} not found in {source}. Available: {available}")
        df = df[df.city.str.lower().isin(wanted)].reset_index(drop=True)
    emb_cols = [c for c in df.columns if c.startswith("emb_")]
    return df, df[emb_cols].to_numpy(dtype=np.float64), emb_cols


def run_tag(source, cities=None):
    """Name used for the saved model and figures: 'streetview', or 'streetview_miami' for a CITY_FILTER run."""
    if not cities:
        return source
    if isinstance(cities, str):
        cities = [cities]
    return source + "_" + "_".join(sorted(slugify(c) for c in cities))


def som_iterations(n):
    return int(max(5000, 20 * n))


def train_som(X, rows, cols, seed=0, init="random"):
    som = MiniSom(rows, cols, X.shape[1], sigma=max(rows, cols) / 2, learning_rate=SOM_LEARNING_RATE,
                  neighborhood_function="gaussian", topology="rectangular", activation_distance="euclidean",
                  decay_function="linear_decay_to_zero", sigma_decay_function="linear_decay_to_one", random_seed=seed)
    if init == "pca":
        som.pca_weights_init(X)            # deterministic start along the two main directions of the data
    else:
        som.random_weights_init(X)         # start from randomly chosen images
    som.train(X, som_iterations(len(X)), random_order=True)
    return som


def bmu_nodes(som, X):
    cols = som.get_weights().shape[1]
    return np.array([r * cols + c for r, c in (som.winner(x) for x in X)])


def cluster_nodes(som, k):
    """Merge neighbouring SOM nodes into k connected clusters (Ward linkage on the node weights)."""
    W = som.get_weights()
    rows, cols, dim = W.shape
    k = int(min(k, rows * cols))
    if k < 2:
        return np.zeros(rows * cols, dtype=int)
    model = AgglomerativeClustering(n_clusters=k, linkage="ward", connectivity=grid_to_graph(rows, cols))
    return model.fit_predict(W.reshape(rows * cols, dim))


def cluster_scores(X, labels):
    if 2 <= len(np.unique(labels)) < len(X):
        return silhouette_score(X, labels), davies_bouldin_score(X, labels)
    return np.nan, np.nan


def evaluate_som(som, X, k=K_EVAL):
    node = bmu_nodes(som, X)
    n_nodes = som.get_weights().shape[0] * som.get_weights().shape[1]
    labels = cluster_nodes(som, k)[node]
    sil, dbi = cluster_scores(X, labels)
    metrics = dict(qe=som.quantization_error(X), te=som.topographic_error(X),
                   empty_nodes=float((np.bincount(node, minlength=n_nodes) == 0).mean()),
                   silhouette=sil, davies_bouldin=dbi)
    return metrics, node, labels


def rule_of_thumb_nodes(n):
    """Classic heuristic (Vesanto, 2000): about 5 * sqrt(n) nodes for n samples."""
    return 5 * math.sqrt(n)


def candidate_sizes(n):
    """Square maps from 2x2 up to ~n/2 nodes, plus the largest square within the rule of thumb."""
    rule = max(2, int(math.sqrt(rule_of_thumb_nodes(n))))
    return sorted({s for s in [2, 3, 4, 5, 6, 8, 10, 12, 14, rule] if s * s <= max(4, n // 2)})

In [ ]:
# 3.2 Choose the data
SOM_SOURCE = "streetview" if any((EMB_DIR / "streetview").glob("*.csv")) else "archdaily"   # or "satellite"
CITY_FILTER = None            # e.g. ["Miami", "Tampa"]; None = all cities of this source

DF, X, EMB_COLS = load_embeddings(SOM_SOURCE, CITY_FILTER)
TAG = run_tag(SOM_SOURCE, CITY_FILTER)
print(f"SOM input: {SOM_SOURCE}, {X.shape[0]} images × {X.shape[1]} embedding dimensions  (results saved as '{TAG}')")
display(DF.groupby("city").size().rename("images").to_frame().T)

## 3.3 Size sweep: train many SOMs and measure them
For every candidate size we train `len(SOM_SEEDS)` maps with different random starts and record the metrics. The two **ARI** columns compare the groupings of the different seeds with each other (nodes, and `K_EVAL` clusters). This takes 1–3 minutes.

In [ ]:
# 3.3 Size sweep
MAX_TOPO_ERROR = 0.05      # a reliable map is smooth: at most 5 % topographic errors,
MIN_STABILITY = 0.50       # its clusters are reproducible across seeds (ARI >= 0.5),
MIN_NODE_STABILITY = 0.50  # and so is the assignment of images to nodes (ARI >= 0.5)


def som_size_sweep(X, sizes, seeds=SOM_SEEDS):
    rows = []
    for s in tqdm(sizes, desc="SOM sizes"):
        nodes, clusters = [], []
        for seed in seeds:
            som = train_som(X, s, s, seed=seed)
            metrics, node, labels = evaluate_som(som, X)
            rows.append(dict(side=s, size=f"{s}×{s}", seed=seed, **metrics))
            nodes.append(node)
            clusters.append(labels)
        pairs = list(itertools.combinations(range(len(seeds)), 2))
        for r in rows[-len(seeds):]:
            r["ari_nodes"] = np.mean([adjusted_rand_score(nodes[i], nodes[j]) for i, j in pairs]) if pairs else np.nan
            r["ari_clusters"] = np.mean([adjusted_rand_score(clusters[i], clusters[j]) for i, j in pairs]) if pairs else np.nan
    return pd.DataFrame(rows)


def suggest_size(sweep, n):
    """The LARGEST reliable map that is not bigger than the rule of thumb (more nodes = more detail)."""
    m = sweep.groupby("side")[["qe", "te", "empty_nodes", "silhouette", "ari_nodes", "ari_clusters"]].mean()
    reliable = ((m.te <= MAX_TOPO_ERROR) & (m.ari_clusters.fillna(0) >= MIN_STABILITY)
                & (m.ari_nodes.fillna(0) >= MIN_NODE_STABILITY))
    within_rule = pd.Series(m.index.to_numpy() ** 2 <= rule_of_thumb_nodes(n), index=m.index)
    if (reliable & within_rule).any():
        side = int(m.index[reliable & within_rule].max())
    elif reliable.any():
        side = int(m.index[reliable].min())
    else:                                   # nothing reliable: the most stable size within the rule of thumb
        pool = m[within_rule] if within_rule.any() else m
        side = int(pool.ari_clusters.fillna(-1).idxmax())
    return side, reliable, within_rule


def sweep_table(sweep, n, reliable, within_rule):
    t = sweep.groupby("side").mean(numeric_only=True).drop(columns=["seed"]).round(3)
    t.insert(0, "images_per_node", (n / t.index.to_numpy() ** 2).round(1))
    t["reliable"] = np.where(reliable.reindex(t.index).to_numpy(), "yes", "no")
    t["within_rule_of_thumb"] = np.where(within_rule.reindex(t.index).to_numpy(), "yes", "no")
    t.index = [f"{s}×{s}" for s in t.index]
    return t


def plot_sweep(sweep, tag, chosen=None):
    metrics = [("qe", "quantization error ↓"), ("te", "topographic error ↓"), ("empty_nodes", "empty nodes ↓"),
               ("silhouette", f"silhouette (k={K_EVAL}) ↑"), ("davies_bouldin", "Davies–Bouldin ↓"),
               ("ari_nodes", "stability of nodes (ARI) ↑"), ("ari_clusters", "stability of clusters (ARI) ↑")]
    g = sweep.groupby("side")
    fig, axes = plt.subplots(1, len(metrics), figsize=(3.1 * len(metrics), 3.2))
    for ax, (col, title) in zip(axes, metrics):
        mean, std = g[col].mean(), g[col].std().fillna(0)
        ax.errorbar(mean.index, mean.values, yerr=std.values, marker="o", capsize=3, color="#4c72b0")
        if chosen is not None:
            ax.axvline(chosen, color="crimson", ls="--", lw=1)
        ax.set_title(title, fontsize=9); ax.set_xlabel("map side (n × n)")
    fig.suptitle(f"{tag}: SOM quality vs map size (mean ± std over {sweep.seed.nunique()} seeds)", fontsize=11)
    plt.tight_layout()
    fig.savefig(SOM_FIG_DIR / f"{tag}_size_sweep.png", dpi=110, bbox_inches="tight")
    plt.show()


SIZES = candidate_sizes(len(X))
t0 = time.time()
SWEEP = som_size_sweep(X, SIZES)
SUGGESTED_SIDE, RELIABLE, WITHIN_RULE = suggest_size(SWEEP, len(X))
print(f"{len(SWEEP)} SOMs trained in {time.time() - t0:.0f} s. Rule of thumb for n = {len(X)}: "
      f"about 5·√n = {rule_of_thumb_nodes(len(X)):.0f} nodes ≈ {math.sqrt(rule_of_thumb_nodes(len(X))):.1f} × {math.sqrt(rule_of_thumb_nodes(len(X))):.1f}")
display(sweep_table(SWEEP, len(X), RELIABLE, WITHIN_RULE))
plot_sweep(SWEEP, TAG, SUGGESTED_SIDE)
print(f"Suggested map size: {SUGGESTED_SIDE} × {SUGGESTED_SIDE} = the largest reliable map within the rule of thumb")

**How the size is suggested.** Small maps are always stable — but they lump very different images together. Large maps fit better (lower QE) but end up with only one or two images per node and change from seed to seed. So:
1. a classic rule of thumb (Vesanto, 2000) says a map should have about **5·√n nodes** for *n* images (≈ 150 nodes = 12 × 12 for 900 images) — this is the **upper limit**;
2. a size is **reliable** if the map is smooth (topographic error ≤ 5 %) and reproducible: both the images-to-nodes assignment and the clusters agree across seeds (ARI ≥ 0.5);
3. we suggest the **largest reliable map within the rule of thumb**. Often that is simply the rule of thumb — the sweep is the **check** that it can be trusted; if it cannot, a smaller size is chosen.

*Empty nodes* are reported but not used to reject a size: on a SOM, empty nodes often sit **between** groups of images and mark a border (look for them on the U-matrix below).

✏️ **Discuss before accepting:** do the curves agree with the suggestion? You may override it below, e.g. `SOM_SIDE = 6`.
Note that silhouette values are low (often 0.05–0.2): image features form a **continuum** rather than well-separated islands — one reason a SOM, which lays out the continuum on a map, is a good tool here.

In [ ]:
# 3.4 Final SOM + choice of the number of clusters + stability check
SOM_SIDE = "auto"          # or an integer, e.g. 8
N_CLUSTERS = "auto"        # or an integer between 2 and 10
STABILITY_SEEDS = [101, 102, 103, 104, 105]


def relabel_by_size(node_cluster, labels):
    """Cluster 0 = the cluster with most images, 1 = the next one, …"""
    order = pd.Series(labels).value_counts().index.tolist()
    order += [c for c in np.unique(node_cluster) if c not in order]
    remap = {old: new for new, old in enumerate(order)}
    return np.array([remap[c] for c in node_cluster]), np.array([remap[c] for c in labels])


def fit_final_som(source, df, X, emb_cols, side, n_clusters="auto", tag=None):
    som = train_som(X, side, side, seed=SEED, init="pca")
    node = bmu_nodes(som, X)
    k_rows = []
    for k in range(2, min(10, side * side - 1) + 1):
        sil, dbi = cluster_scores(X, cluster_nodes(som, k)[node])
        k_rows.append(dict(k=k, silhouette=sil, davies_bouldin=dbi))
    k_table = pd.DataFrame(k_rows)
    if n_clusters == "auto":
        pool = k_table[k_table.k >= 3] if (k_table.k >= 3).any() else k_table
        k = int(pool.loc[pool.silhouette.idxmax(), "k"]) if pool.silhouette.notna().any() else int(pool.k.iloc[0])
    else:
        k = int(min(n_clusters, side * side))
    node_cluster = cluster_nodes(som, k)
    node_cluster, labels = relabel_by_size(node_cluster, node_cluster[node])
    # stability: train the same configuration from other random starts and compare the groupings
    ari = []
    for seed in STABILITY_SEEDS:
        other = train_som(X, side, side, seed=seed)
        ari.append(adjusted_rand_score(labels, cluster_nodes(other, k)[bmu_nodes(other, X)]))
    W = som.get_weights().reshape(side * side, -1)
    sil, dbi = cluster_scores(X, labels)
    return dict(source=source, tag=tag or source, df=df.reset_index(drop=True), X=X, emb_cols=emb_cols, som=som,
                side=side, node=node, node_cluster=node_cluster, labels=labels, k=k, k_table=k_table,
                bmu_dist=np.linalg.norm(X - W[node], axis=1), stability=np.array(ari),
                metrics=dict(qe=float(som.quantization_error(X)), te=float(som.topographic_error(X)),
                             silhouette=float(sil), davies_bouldin=float(dbi),
                             stability_ari_mean=float(np.mean(ari)), stability_ari_std=float(np.std(ari))))


def plot_k_choice(res):
    t = res["k_table"]
    fig, ax = plt.subplots(figsize=(6, 3))
    ax.plot(t.k, t.silhouette, "o-", label="silhouette ↑")
    ax2 = ax.twinx(); ax2.plot(t.k, t.davies_bouldin, "s--", color="#dd8452", label="Davies–Bouldin ↓")
    ax.axvline(res["k"], color="crimson", ls="--")
    ax.set_xlabel("number of clusters k"); ax.set_ylabel("silhouette"); ax2.set_ylabel("Davies–Bouldin")
    ax.set_title(f"{res['tag']}: choosing k (chosen k = {res['k']})")
    fig.legend(loc="upper right", fontsize=8, bbox_to_anchor=(0.88, 0.88))
    plt.tight_layout(); plt.show()


def describe_stability(res):
    a = res["stability"].mean()
    verdict = "very stable" if a > 0.8 else "moderately stable" if a > 0.5 else "unstable — interpret with care"
    print(f"Stability: ARI with {len(res['stability'])} re-trained maps = {a:.2f} ± {res['stability'].std():.2f} -> {verdict}")


side = SUGGESTED_SIDE if SOM_SIDE == "auto" else int(SOM_SIDE)
RES = fit_final_som(SOM_SOURCE, DF, X, EMB_COLS, side, N_CLUSTERS, tag=TAG)
plot_k_choice(RES)
print(f"final map {RES['side']}×{RES['side']}, k = {RES['k']}:", {k: round(v, 3) for k, v in RES["metrics"].items()})
describe_stability(RES)

## 3.5 Read the map
**(a) U-matrix and clusters.** Left: the U-matrix — the average distance between each node and its neighbours. Dark cells are "walls" between groups of similar images; red dots show how many images each node holds. Right: the clusters (colours) and the number of images per node.

In [ ]:
# 3.5a U-matrix + cluster map
def plot_umatrix(res, save=True):
    side = res["side"]
    U = res["som"].distance_map(scaling="mean")          # average distance to the neighbouring nodes
    hits = np.bincount(res["node"], minlength=side * side).reshape(side, side)
    C = res["node_cluster"].reshape(side, side)
    cmap = mpl.colors.ListedColormap(CLUSTER_COLORS[:max(res["k"], int(C.max()) + 1)])
    fig, axes = plt.subplots(1, 2, figsize=(12, 5.4))
    im = axes[0].imshow(U, cmap="bone_r")
    plt.colorbar(im, ax=axes[0], fraction=0.046, label="mean distance to neighbouring nodes (scaled)")
    yy, xx = np.nonzero(hits)
    axes[0].scatter(xx, yy, s=hits[yy, xx] / hits.max() * 250, c="tab:red", alpha=0.6)
    axes[0].set_title("U-matrix (dark = border) · dots = images per node")
    axes[1].imshow(C, cmap=cmap, vmin=-0.5, vmax=cmap.N - 0.5)
    for i in range(side):
        for j in range(side):
            axes[1].text(j, i, hits[i, j] if hits[i, j] else "", ha="center", va="center", fontsize=8)
    axes[1].set_title(f"{res['k']} clusters · numbers = images per node")
    for ax in axes:
        ax.set_xticks(range(side)); ax.set_yticks(range(side)); ax.tick_params(labelsize=7)
    fig.suptitle(f"{res['tag']} — {side}×{side} SOM", fontsize=12)
    plt.tight_layout()
    if save:
        fig.savefig(SOM_FIG_DIR / f"{res['tag']}_umatrix_clusters.png", dpi=110, bbox_inches="tight")
    plt.show()


plot_umatrix(RES)

**(b) The map as images.** Each node shows its most typical image (the one closest to the node's weights); the frame colour is the cluster and the small number is how many images the node holds.

In [ ]:
# 3.5b One representative image per node
def plot_som_images(res, thumb=110, save=True):
    side, df, X = res["side"], res["df"], res["X"]
    W = res["som"].get_weights()
    fig, axes = plt.subplots(side, side, figsize=(1.35 * side + 1, 1.35 * side + 1), squeeze=False)
    for i in range(side):
        for j in range(side):
            ax = axes[i, j]
            ax.set_xticks([]); ax.set_yticks([])
            color = CLUSTER_COLORS[res["node_cluster"][i * side + j] % len(CLUSTER_COLORS)]
            for sp in ax.spines.values():
                sp.set_visible(True); sp.set_edgecolor(color); sp.set_linewidth(3)
            members = np.where(res["node"] == i * side + j)[0]
            if len(members):
                best = members[np.argmin(np.linalg.norm(X[members] - W[i, j], axis=1))]
                im = load_rgb(IMG_DIR / df.relpath.iloc[best]); im.thumbnail((thumb, thumb))
                ax.imshow(im)
                ax.set_title(len(members), fontsize=7, pad=1)
            else:
                ax.set_facecolor("#f0f0f0")
    fig.suptitle(f"{res['tag']}: most typical image of every node (frame = cluster)", fontsize=11)
    plt.subplots_adjust(wspace=0.08, hspace=0.25)
    if save:
        fig.savefig(SOM_FIG_DIR / f"{res['tag']}_som_images.png", dpi=110, bbox_inches="tight")
    plt.show()


plot_som_images(RES)

**(c) Component planes.** The average of a readable indicator for the images of each node. Compare with the cluster map: is one cluster "the green one"? Where are the dense, high-edge streets?

In [ ]:
# 3.5c Component planes of the readable indicators
def plot_component_planes(res, cols=None, save=True):
    side, df = res["side"], res["df"]
    cols = [c for c in (cols or INT_COLS_BY_SOURCE.get(res["source"], [])) if c in df.columns]
    if not cols:
        print("No indicator columns found.")
        return
    means = df[cols].groupby(res["node"]).mean().reindex(range(side * side))
    cmap = mpl.colormaps["viridis"].copy()
    cmap.set_bad("#d9d9d9")                               # empty nodes in grey
    ncols = 4
    nrows = math.ceil(len(cols) / ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(3.6 * ncols, 3.2 * nrows), squeeze=False)
    for ax in axes.ravel():
        ax.axis("off")
    for ax, c in zip(axes.ravel(), cols):
        im = ax.imshow(means[c].to_numpy().reshape(side, side), cmap=cmap)
        ax.set_title(c.replace("int_", ""), fontsize=10); ax.axis("on"); ax.set_xticks([]); ax.set_yticks([])
        plt.colorbar(im, ax=ax, fraction=0.046)
    fig.suptitle(f"{res['tag']}: component planes (grey = empty node)", fontsize=11)
    plt.tight_layout()
    if save:
        fig.savefig(SOM_FIG_DIR / f"{res['tag']}_component_planes.png", dpi=110, bbox_inches="tight")
    plt.show()


plot_component_planes(RES)

**(d) Cluster profiles and cities.** Left: how each cluster differs from the average image (z-scores of the indicators: red = more than average). Right: which share of each city falls into each cluster.

In [ ]:
# 3.5d Cluster profiles + city composition
def plot_cluster_profiles(res, save=True):
    df, labels = res["df"], res["labels"]
    cols = [c for c in df.columns if c.startswith("int_") and df[c].std() > 0]
    prof = df[cols].groupby(labels).mean()
    z = (prof - df[cols].mean()) / df[cols].std()
    sizes = pd.Series(labels).value_counts().sort_index()
    tab = pd.crosstab(pd.Series(labels, name="cluster"), df.city.rename("city"))
    tab = tab[[c for c in CITY_ORDER if c in tab.columns] + [c for c in tab.columns if c not in CITY_ORDER]]
    within_city = tab / tab.sum(axis=0)
    fig, axes = plt.subplots(1, 2, figsize=(16, 0.45 * len(prof) + 2.6), gridspec_kw={"width_ratios": [1.6, 1]})
    im = axes[0].imshow(z.values, cmap="RdBu_r", vmin=-2, vmax=2, aspect="auto")
    axes[0].set_xticks(range(len(cols))); axes[0].set_xticklabels([c.replace("int_", "") for c in cols], rotation=60, ha="right", fontsize=8)
    axes[0].set_yticks(range(len(prof))); axes[0].set_yticklabels([f"cluster {c} (n={sizes[c]})" for c in prof.index])
    plt.colorbar(im, ax=axes[0], fraction=0.03, label="z-score vs all images")
    axes[0].set_title("cluster profiles")
    im = axes[1].imshow(within_city.values, cmap="Blues", vmin=0, aspect="auto")
    axes[1].set_xticks(range(tab.shape[1])); axes[1].set_xticklabels(tab.columns, rotation=60, ha="right", fontsize=8)
    axes[1].set_yticks(range(tab.shape[0])); axes[1].set_yticklabels([f"cluster {c}" for c in tab.index])
    for (r, c), v in np.ndenumerate(within_city.values):
        axes[1].text(c, r, f"{v:.0%}", ha="center", va="center", fontsize=7, color="white" if v > 0.5 else "black")
    axes[1].set_title("share of each city's images in each cluster")
    plt.tight_layout()
    if save:
        fig.savefig(SOM_FIG_DIR / f"{res['tag']}_cluster_profiles.png", dpi=110, bbox_inches="tight")
    plt.show()


plot_cluster_profiles(RES)

**(e) Look inside a cluster.** The images closest to the cluster centre first. Change `SHOW_CLUSTER`.

In [ ]:
# 3.5e Gallery of one cluster
SHOW_CLUSTER = 0


def show_cluster(res, cluster, n=20):
    idx = np.where(res["labels"] == cluster)[0]
    if len(idx) == 0:
        print(f"Cluster {cluster} is empty (clusters go from 0 to {res['k'] - 1}).")
        return
    centre = res["X"][idx].mean(axis=0)
    order = idx[np.argsort(np.linalg.norm(res["X"][idx] - centre, axis=1))][:n]
    df = res["df"]
    show_image_grid([IMG_DIR / p for p in df.relpath.iloc[order]],
                    [f"{c} · {str(u)[:18]}" for c, u in zip(df.city.iloc[order], df.unit.iloc[order])],
                    ncols=5, size=2.6, suptitle=f"{res['tag']} — cluster {cluster}: {len(idx)} images (most typical first)")


show_cluster(RES, SHOW_CLUSTER)

**(f) Where are the clusters in the cities?** Each dot is one image location coloured by its cluster (for ArchDaily: the project location); use the layer control to show one cluster at a time.

In [ ]:
# 3.5f Clusters on the map
def cluster_map(res):
    df = res["df"]
    if df.lat.isna().all():
        print(f"{res['tag']}: the images have no coordinates -> no map.")
        return None
    m = base_map()
    for c in range(res["k"]):
        fg = folium.FeatureGroup(name=f"cluster {c}")
        for r in df[(res["labels"] == c) & df.lat.notna().to_numpy()].itertuples():
            folium.CircleMarker([r.lat, r.lon], radius=4, color=CLUSTER_COLORS[c % 10], fill=True, fill_opacity=0.9,
                                weight=1, tooltip=f"cluster {c} · {r.city} · {r.unit} · {r.image_id}").add_to(fg)
        fg.add_to(m)
    folium.LayerControl(collapsed=False).add_to(m)
    add_legend(m, {f"cluster {c}": CLUSTER_COLORS[c % 10] for c in range(res["k"])}, f"{res['tag']} SOM clusters")
    return m


cmap_ = cluster_map(RES)
if cmap_ is not None:
    display(cmap_)

## 3.6 Save: labels into the CSVs, and the SOM model
* The CSVs of Section 2 get five new columns: `som_row`, `som_col`, `som_node` (= row × side + col), `som_cluster`, `som_bmu_dist` (distance to the node; large = atypical image). Re-running replaces them.
  A SOM trained with `CITY_FILTER` does **not** touch these columns: its labels go to `03_som/<tag>_labels.csv`, so the all-cities results stay consistent.
* The trained SOM, the node → cluster table and the settings are saved with `pickle` in `03_som/models/<tag>_som.pkl` (+ the weights as `.npy` and a readable `.json`, which also records the package versions — re-load the pickle with the same `minisom` version). Together with `<source>_fusion_pca.joblib` from Section 2 you can later assign **new images** to the same clusters without retraining.

In [ ]:
# 3.6a Write labels and save the model
def som_label_table(res):
    side = res["side"]
    return pd.DataFrame({"image_id": res["df"].image_id.astype(str), "som_row": res["node"] // side,
                         "som_col": res["node"] % side, "som_node": res["node"], "som_cluster": res["labels"],
                         "som_bmu_dist": res["bmu_dist"].round(4)})


def write_som_labels(res):
    lab = som_label_table(res)
    if res["tag"] != res["source"]:                    # CITY_FILTER run: separate file, CSVs untouched
        path = SOM_DIR / f"{res['tag']}_labels.csv"
        lab.merge(res["df"][["image_id", "city", "unit"]].astype({"image_id": str}), on="image_id").to_csv(path, index=False)
        print(f"  CITY_FILTER run -> labels saved to {path.relative_to(BASE_DIR)} (the per-city CSVs keep the all-cities labels)")
        return
    for f in sorted((EMB_DIR / res["source"]).glob("*.csv")):
        df = pd.read_csv(f)
        df = df.drop(columns=[c for c in SOM_COLS if c in df.columns])
        df["image_id"] = df.image_id.astype(str)
        df = df.merge(lab, on="image_id", how="left")
        df.to_csv(f, index=False, float_format="%.8g")
        print(f"  {f.relative_to(BASE_DIR)}: {df.som_cluster.notna().sum()} labelled rows")


def save_som(res):
    bundle = dict(som=res["som"], side=res["side"], node_cluster=res["node_cluster"], n_clusters=res["k"],
                  source=res["source"], tag=res["tag"], cities=sorted(set(res["df"].city)), emb_columns=res["emb_cols"],
                  metrics=res["metrics"], fusion_model=f"{res['source']}_fusion_pca.joblib",
                  versions=dict(minisom=package_version("minisom"), numpy=np.__version__,
                                sklearn=sklearn.__version__, python=sys.version.split()[0]),
                  created=datetime.now().isoformat(timespec="seconds"))
    path = SOM_MODEL_DIR / f"{res['tag']}_som.pkl"
    with open(path, "wb") as fh:
        pickle.dump(bundle, fh)
    np.save(SOM_MODEL_DIR / f"{res['tag']}_som_weights.npy", res["som"].get_weights())
    readable = {k: v for k, v in bundle.items() if k not in ("som", "node_cluster")}
    readable["node_cluster"] = res["node_cluster"].reshape(res["side"], res["side"]).tolist()
    (SOM_MODEL_DIR / f"{res['tag']}_som.json").write_text(json.dumps(readable, indent=1, default=str))
    print(f"  model saved -> {path.relative_to(BASE_DIR)}")


write_som_labels(RES)
save_som(RES)

In [ ]:
# 3.6b Re-load the saved SOM and check that it reproduces the saved labels
with open(SOM_MODEL_DIR / f"{TAG}_som.pkl", "rb") as fh:
    loaded = pickle.load(fh)

check_df, check_X, _ = load_embeddings(loaded["source"], None if loaded["tag"] == loaded["source"] else loaded["cities"])
if loaded["tag"] == loaded["source"]:
    stored = check_df.som_cluster.to_numpy()
else:
    stored = (check_df[["image_id"]].astype(str)
              .merge(pd.read_csv(SOM_DIR / f"{TAG}_labels.csv", dtype={"image_id": str}), on="image_id", how="left")
              .som_cluster.to_numpy())
pred_cluster = loaded["node_cluster"][bmu_nodes(loaded["som"], check_X[:, :len(loaded["emb_columns"])])]
print(f"Loaded '{loaded['tag']}': a {loaded['side']}×{loaded['side']} SOM with {loaded['n_clusters']} clusters, "
      f"trained on {loaded['cities']} (saved with {loaded['versions']})")
print(f"Labels reproduced for {(pred_cluster == stored).mean():.0%} of the {len(check_df)} images")

### 3.6c Using the saved models on a new image
This is how you would place a **new** photo on the map: the same four lenses → the saved fusion + PCA pipeline → the saved SOM. Here we use an image that is already in the data set, so we can check that we get its stored cluster back.
(Tiny numerical differences between processing one image and a batch of 16 can occasionally move a borderline image to the neighbouring node.)

In [ ]:
# 3.6c Predict the SOM node / cluster of any image file
def predict_image(path, source, tag=None):
    needed = ["prepare_image", "vgg_features", "seg_labelmaps", "seg_features", "edge_analysis", "color_analysis"]
    missing = [f for f in needed if f not in globals()]
    if missing:
        raise RuntimeError(f"Run the cells 2.0-2.5 first (missing: {missing}).")
    img = prepare_image(path, source)
    raw = {"vgg": vgg_features([img]), "seg": seg_features(seg_labelmaps([img])),
           "edge": edge_analysis(img)[0][None], "color": color_analysis(img)[0][None]}
    pipe = joblib.load(FEAT_MODEL_DIR / f"{source}_fusion_pca.joblib")
    with open(SOM_MODEL_DIR / f"{tag or source}_som.pkl", "rb") as fh:
        bundle = pickle.load(fh)
    long_vector = np.hstack([raw[b] for b in ["vgg", "seg", "edge", "color"]]).astype(np.float64)
    z = pipe.transform(long_vector)[0]
    r, c = bundle["som"].winner(z)
    node = r * bundle["side"] + c
    return dict(som_row=int(r), som_col=int(c), som_node=int(node), som_cluster=int(bundle["node_cluster"][node]))


row = check_df.iloc[0]
if "vgg_features" in globals():
    pred = predict_image(IMG_DIR / row.relpath, SOM_SOURCE, TAG)
    print("image:", row.relpath)
    print("predicted:", pred)
    print("stored cluster:", int(stored[0]))
else:
    print("To try predict_image after a disconnect, first run the cells 2.0-2.5 (feature extractors). Skipped.")

## 3.7 The other sources
The walkthrough above used `SOM_SOURCE`. The next cell runs the **same procedure** (size sweep → suggested size → final SOM → automatic k → stability → figures → labels → saved model) for the remaining sources, on all their cities, so that every CSV gets its labels. The figures are also saved in `03_som/figures/`.

In [ ]:
# 3.7 Run the whole SOM procedure for the other sources
def run_som_pipeline(source, side="auto", n_clusters="auto"):
    df, X, emb_cols = load_embeddings(source)
    if len(X) < 8:
        print(f"{source}: only {len(X)} images -> SOM skipped")
        return None
    print(f"\n==================== {source}: {len(X)} images ====================")
    sweep = som_size_sweep(X, candidate_sizes(len(X)))
    best_side, reliable, within_rule = suggest_size(sweep, len(X))
    display(sweep_table(sweep, len(X), reliable, within_rule))
    plot_sweep(sweep, source, best_side)
    res = fit_final_som(source, df, X, emb_cols, best_side if side == "auto" else int(side), n_clusters)
    print(f"map {res['side']}×{res['side']}, k = {res['k']}:", {k: round(v, 3) for k, v in res["metrics"].items()})
    describe_stability(res)
    plot_umatrix(res)
    plot_som_images(res)
    plot_cluster_profiles(res)
    write_som_labels(res)
    save_som(res)
    return res


OTHER_RESULTS = {}
for src in SOURCES:
    if (src != SOM_SOURCE or TAG != SOM_SOURCE) and any((EMB_DIR / src).glob("*.csv")):
        if src == SOM_SOURCE:
            print(f"(3.2–3.6 used CITY_FILTER, so {src} is also run on all its cities here)")
        OTHER_RESULTS[src] = run_som_pipeline(src)

---
# 4 · Wrap-up: what is now in your Drive?

In [ ]:
# 4.1 Checklist of the outputs
def inventory():
    rows = []
    for src in SOURCES:
        csvs = sorted((EMB_DIR / src).glob("*.csv"))
        labelled = bool(csvs) and all("som_cluster" in pd.read_csv(f, nrows=1).columns for f in csvs)
        rows.append({"source": src,
                     "images": len(list((IMG_DIR / src).glob("*/*.jpg"))),
                     "city folders": len([p for p in (IMG_DIR / src).glob("*") if p.is_dir()]),
                     "raw features (.npz)": len(list((RAW_DIR / src).glob("*.npz"))),
                     "embedding CSVs": len(csvs),
                     "SOM labels in CSVs": "yes" if labelled else "no",
                     "fusion+PCA model": "yes" if (FEAT_MODEL_DIR / f"{src}_fusion_pca.joblib").exists() else "no",
                     "SOM model": "yes" if (SOM_MODEL_DIR / f"{src}_som.pkl").exists() else "no"})
    display(pd.DataFrame(rows).set_index("source"))
    extra = sorted(p.name for p in SOM_MODEL_DIR.glob("*_som.pkl") if p.stem[:-4] not in SOURCES)
    if extra:
        print("additional SOMs from CITY_FILTER runs:", extra)
    total = sum(f.stat().st_size for f in BASE_DIR.rglob("*") if f.is_file()) / 1e6
    print(f"Week8 folder: {total:.0f} MB in {BASE_DIR}")
    print("figures:", sorted(p.name for p in list(FEAT_FIG_DIR.glob("*.png")) + list(SOM_FIG_DIR.glob("*.png"))))


inventory()

## What you have now
| In Drive | Content |
|---|---|
| `01_images/archdaily/<city>/` · `satellite/<city>/` · `streetview/<city>/` | the images, plus manifests (`*_manifest.csv`) and the sampling plan |
| `02_features/raw/<source>/<city>.npz` | the long raw vectors (VGG 512 + segmentation 150 + edges 81 + colour 89 = 832 numbers) with column names |
| `02_features/embeddings/<source>/<city>.csv` | one row per image: ids, location, embedding `emb_…`, indicators `int_…`, **SOM labels** `som_…` |
| `02_features/models/<source>_fusion_pca.joblib` | the fitted fusion + PCA pipeline |
| `03_som/models/<source>_som.pkl` | the trained SOM + node → cluster table (+ `.npy` weights, `.json` summary) |
| `02_features/figures/`, `03_som/figures/` | all the figures |

## Exercises (homework, or in class if time is left — pick two)
1. **Lens weights.** Set `BLOCK_WEIGHTS = {"vgg": 0, "seg": 1, "edge": 1, "color": 1}` (no deep features), then re-run **every cell from 2.9a to 2.11** and then **all of Section 3**. Are the clusters easier to *explain*? Are they more or less stable?
2. **Embedding size.** Same procedure with `N_COMPONENTS = 10` and then `N_COMPONENTS = 100` (2.9b → 2.11 → Section 3): what happens to the SOM metrics and to the cluster galleries?
3. **One city vs all.** Set `CITY_FILTER = ["Miami"]` in 3.2 and re-run 3.2 → 3.6. The Miami-only SOM is saved as `streetview_miami_som.pkl` and its labels in `03_som/streetview_miami_labels.csv`, next to the Florida-wide results. Does it find finer distinctions?
4. **Cross-view.** Satellite and street view share `image_id`. Merge the two CSVs on `image_id` and cross-tabulate `som_cluster` of both: do "green streets" (street view) coincide with "green blocks" (satellite)?
5. **Your own site** *(costs a few Google requests)*. Add a unit for a neighbourhood you know to `CITY_UNITS`, re-run Section 1: the city's quota is shared with the new unit, so each existing unit gives up a few points (their images go to `_archive/`), and only the new unit's points are downloaded. Then use `predict_image` to place the new images on the existing SOM (before re-running Sections 2–3).